# Plot Az-consistent-forward scalar-amplitude inversion - Nicoya (non-CK plate model)

Counterpart of `plt_syn_slip_az_consistfwd_nicoyaCK_locking.ipynb` for the
**non-CK** plate model (mesh `nicoyaden_sm`). Same scheme, same plotting layout;
only the mesh / projection / plate-interface file path / slip-prescription
parameters differ. Result files come from `synth_*_az_consistfwd_*_nicoya.py`.

* Inverts a single scalar amplitude `a(x)` in [0, amp_max] with the slip vector
  direction fixed by the local plate-convergence azimuth (per-facet `(c_strike, c_dip)`).
* Forward and inversion share the same `PDEVarf_AzSlip_Direct` operator (no spots).
* Recovery ratio = `rec_amp / amp` is the inferred locking ratio.
* The 5 (forward, inverted) mu combos: H_H, S_S, S_H, 3D_3D, 3D_H.


In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'


In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/synth/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
flag_savefig = True
# flag_savefig = False

# flag to indicate the inversion scheme
inv_scheme = "_az"   # to indicate that the inversion fixed the azimuth of slip

# Define the inversion results path
resultpath = "syn_slip/"

# define mesh name (non-CK plate model)
# meshname = "nicoya"     # larger fault interface
# meshname = "nicoya2"    # smaller fault interface
# meshname = "nicoya3"    # same as nicoya2 but 5-km mesh size on fault
meshname = "nicoyaden_sm"     # denser mesh, smaller fault zone
# meshname = "nicoyaden_all"  # denser mesh, all subduction interface

print(meshname)

# Read the plate interface (text file → DataFrame with lon/lat/dep)
plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())

# Plate file in GMT .grd format (used directly with grdcontour, the non-CK idiom)
grd_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"

# Trench file path (parsed in cell 4 with utp.parse_trench_file)
trench_file = "plateinterface/trench.gmt.inuse"

# anaysis type, locking or coseismic?
type = "locking"  # "locking" or "coseismic"
# type = "coseismic"

# whethere the synthetics are polluted
pollute = True
print(pollute)

# noise std type, either 'uniform' or 'datastd'
pollute_type = 'uniform'
print(pollute_type)

m2mm = 1e3   # meter to mm
km2m = 1e3   # km to m

if type == "locking":
       obs_file = "data/Kyriakopoulos_novolcano.csv"
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                            'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                            'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
       df['lon'] = -1*df['lon']
       cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                     'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
       df[cols_to_convert] = df[cols_to_convert] / m2mm

       if pollute:
              if pollute_type == 'uniform':
                     noise_std_h = 0.5 * (df['vx_std_Car'].mean() + df['vy_std_Car'].mean())
                     noise_std_v = df['vz_std_Car'].mean()
                     error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v
              elif pollute_type == 'datastd':
                     error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
       else:
              error_e, error_n, error_v = df['vx_std_Car']*0, df['vy_std_Car']*0, df['vz_std_Car']*0

else:
       obs_file = "data/Protti_etal_2014_tableS1.csv"
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                            'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])
       if pollute:
              noise_std_h = 0.5 * (df['ux_std'].mean() + df['uy_std'].mean())
              error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_h
       else:
              error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())


In [ ]:
# Az-consistent-forward scheme is always bounded: amplitude is internally
# constrained to [0, amp_max] inside solveCoseismicInversion_AzSlip.
# No BOUNDED toggle is needed here.


In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv"
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1,
                      names=['id', 'lat', 'lon', 'elv'])
print(volcano.head())

region  = [-87, -84, 8.5, 11.5]
region1 = [-86.75, -84.4, 8.75, 11.25]

# Trench segments inside region (non-CK ancestor idiom)
segments = utp.parse_trench_file(datadir + trench_file,
                                 lon_range=(region[0], region[1]),
                                 lat_range=(region[2], region[3]))
trench = utp.segments_to_dataframe(segments)
print(trench.head())

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]

In [ ]:
### Choose the slip model, all models are in m_dip direction, m_strike = 0
# slipmodel = 1     # along-strike 3-stripe pattern, shallow-middle-deep
slipmodel = 2     # along-strike 2-stripe pattern with gap, shallow-deep
# slipmodel = 3     # along-strike 1-stripe pattern, complementary of model 2, middle
# slipmodel = 4     # checkerboard pattern in strike-dip direction
# slipmodel = 5     # checkerboard pattern in x and y direction

# === Az-scheme amp from plate-convergence azimuth (cf scripts) ===
# slip_azimuth_deg  = 45.0    # CW from North; N45E = trench-normal convergence
# slip_azimuth_deg  = 26.0    # CW from North; N26E = oblique
slip_azimuth_deg  = 33.5    # CW from North; N33.5E = mildly oblique convergence

V_norm_mm = 78.5
V_para = 27
V_para_const = 11
if slip_azimuth_deg == 45.0:
    amp = V_norm_mm
elif slip_azimuth_deg == 26.0:
    amp = round(np.sqrt(V_norm_mm**2 + V_para**2))
elif slip_azimuth_deg == 33.5:
    amp = round(np.sqrt(V_norm_mm**2 + (V_para - V_para_const)**2))   # ~80 mm
amp = amp / m2mm   # mm -> m
print(f"amp = {amp*m2mm:.1f} mm/yr  (slip_azimuth_deg = {slip_azimuth_deg})")

# === Build slip_str_gt to match the cf forward script's naming ===
# Template:  _<pattern>_x{x0}_y{y0}_lx{lx}_dx{dx}_rot{rot}_ms{amp:g}_az{az:g}[_pou|_pod]
if slipmodel == 1:
    # 3-stripe shallow-middle-deep
    x_len = 35e3; dx = 25e3; x0 = 0e3; y0 = 0e3
    pattern_rot_deg = 0.0
    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 2:
    # 2-stripe shallow-deep with gap (non-CK uses x_len=90, y0=-40)
    x_len = 90e3
    y_len = 300e3
    dx = 35e3
    stripe_spacing = x_len + dx
    x0 = 0; y0 = -40e3
    pattern_rot_deg = 0.0
    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 3:
    # 1-stripe intermediate (complement of model 2)
    x_len = 40e3; y_len = 300e3; dx = 100e3
    stripe_spacing = x_len + dx
    x0 = 0; y0 = 22.5e3
    pattern_rot_deg = 0.0
    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 4:
    # Checker in strike-dip frame, rotated 45 deg
    dx = 35e3; dy = 35e3; x0 = -20e3; y0 = -45e3
    pattern_rot_deg = 45
    slip_str_gt = (
        f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

elif slipmodel == 5:
    # Checker in x-y (E,N) frame
    dx = 30e3; dy = 30e3; x0 = 20e3; y0 = -10e3
    pattern_rot_deg = 0
    slip_str_gt = (
        f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}"
        f"_rot{pattern_rot_deg:g}_ms{amp:g}_az{slip_azimuth_deg:g}"
    )

slip_str_gt1 = slip_str_gt   # base (no pollution suffix)

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt = slip_str_gt + "_pou"
    elif pollute_type == 'datastd':
        slip_str_gt = slip_str_gt + "_pod"

print("Slip model string:", slip_str_gt)

# === slip_pattern + pattern_option (used by L-curve filenames in cf scripts) ===
if slipmodel in (1, 2, 3):
    slip_pattern = "stripe"
    pattern_option = {1: 0, 2: 1, 3: 2}[slipmodel]
elif slipmodel in (4, 5, 6):
    slip_pattern = "checker"
    pattern_option = {4: 1, 5: 2, 6: 3}[slipmodel]
else:
    slip_pattern, pattern_option = "other", 0
print("slip_pattern:", slip_pattern, " pattern_option:", pattern_option)


In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

mu_b = 0          # 40 GPa
mu_background = mu_expression(mu_b)
mu_l = 0.9730     # ~55 GPa
mu_lower = mu_expression(mu_l)
mu_u = -0.9730    # ~25 GPa
mu_upper = mu_expression(mu_u)
mu_v = 0
mu_volcano = mu_expression(mu_v)

if mu_v == 0:
    homo_str    = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    sw_str      = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    het3d_str_ori = f"_DeShon3D_ref1D_{round(1.0)}_hull"
    contrast_factor = 2.5
    het3d_str   = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"

    CG_mu_deg = 2   # CG2 for 3D model (consistent with cf forward / inversion scripts)
    if CG_mu_deg == 2:
        het3d_str     = het3d_str     + f"_CG{CG_mu_deg}"
        het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
else:
    print("Volcano mu = %.1f, upper mu = %.1f, lower mu = %.1f" % (mu_volcano, mu_upper, mu_lower))
    sw_str   = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(sw_str)
print(het3d_str_ori)
print(het3d_str)


In [ ]:
# Define the reference point (non-CK convention: azimuthal equidistant projection)
lon0, lat0 = -85.5, 10
R = 6371.0   # Earth radius in km

# Load the fault geometry
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                    skiprows=lambda x: x % 3 != 1,
                    names=['xf', 'yf', 'zf'])
loc_f = loc_f / km2m

# Inverse projection (mesh xy -> lon/lat) — non-CK uses azi_equidist (no rotation)
loc_f['lon'], loc_f['lat'] = utp.inverse_azi_equidist_proj(loc_f['xf'], loc_f['yf'], lon0, lat0)
print(loc_f[['lon', 'lat']].head())

# ── Ground-truth slip from cf forward (slip_true_cf_*.txt) ───────────────────
# Columns: xf yf zf m_phys s_strike s_dip slip_mag c_mag (m, m/yr)
outFileName = 'slip_true_cf_' + meshname + slip_str_gt + '.txt'
print(outFileName)
slip_cf_raw = np.loadtxt(datadir + resultpath + outFileName, comments='#')
mtrue_s = pd.DataFrame({
    's_strike': slip_cf_raw[:, 4],   # m/yr (raw component)
    's_dip':    slip_cf_raw[:, 5],   # m/yr
    'mag':      slip_cf_raw[:, 3],   # m_phys, m/yr
})
print('Raw m_phys range (m/yr):', mtrue_s['mag'].min(), mtrue_s['mag'].max())

# Normalize to 0..1 (plotting code multiplies by amp*m2mm to get mm where needed)
if type == 'locking':
    cols_to_convert = ['s_strike', 's_dip', 'mag']
    mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] / amp
print('Normalized mag (locking ratio) range:', mtrue_s['mag'].min(), mtrue_s['mag'].max())


In [ ]:
col2plt = 'mag'

In [ ]:
def plot_slip(mtrue_s, col2plt, region):
    
    # slipsybsz = "c0.04c"
    slipsybsz = "c0.08c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            # maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*1e3
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*1e3, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*1e3, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*1e3, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            

            # fig.plot(x=df['lon'], y=df['lat'], style="s0.1c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(mm)"])

    fig.show()


plot_slip(mtrue_s, col2plt, region)

In [ ]:
# ── Ground-truth moment for cf scheme: moment_true_cf_*.txt ──────────────────
def read_gt_moment(datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'moment_true_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    mom_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                           names=['moment', 'mw', 'potency'])
    return mom_true


# ── Ground-truth station displacement: d_obs_cf_*.txt (per mu model) ────────
# Non-CK mesh frame == geographic frame, so NO rot_xy on (ux, uy).
def read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_true['lon'], u_true['lat'] = df['lon'], df['lat']
    u_true['mag']   = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true


def read_station_grid(datadir, resultpath, meshname, grid_spacing_deg=None):
    if grid_spacing_deg is None:
        outFileName = 'dense_grid_coordinates_' + meshname + '.txt'
    else:
        outFileName = f"dense_grid_coordinates_{meshname}_{grid_spacing_deg}.txt"
    print(outFileName)
    sta_grid = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                           names=['lon', 'lat', 'x', 'y', 'z'])
    cols_to_convert = ['x', 'y', 'z']
    sta_grid[cols_to_convert] = sta_grid[cols_to_convert] * km2m
    return sta_grid


# ── Ground-truth grid displacement: d_obs_grid_cf_*.txt ──────────────────────
# NOTE: grid_spacing concatenated WITHOUT separator (matches cf scripts)
# Non-CK mesh frame == geographic frame, so NO rot_xy.
def read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt,
                      struc_str_for, grid_spacing_deg=None):
    if grid_spacing_deg is None:
        outFileName = 'd_obs_grid_cf_' + meshname + slip_str_gt + struc_str_for + '.txt'
    else:
        outFileName = ('d_obs_grid_cf_' + meshname + slip_str_gt + struc_str_for
                       + str(grid_spacing_deg) + '.txt')
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    u_true['lon'], u_true['lat'] = sta_grid['lon'], sta_grid['lat']
    u_true['mag']   = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()
    return u_true


def build_diff_disp(u_1, u_2):
    u_diff = u_1.copy()
    u_diff['ux']    = u_2['ux']    - u_1['ux']
    u_diff['uy']    = u_2['uy']    - u_1['uy']
    u_diff['uz']    = u_2['uz']    - u_1['uz']
    u_diff['mag']   = u_2['mag']   - u_1['mag']
    u_diff['mag_h'] = u_2['mag_h'] - u_1['mag_h']
    u_diff['mag_v'] = u_2['mag_v'] - u_1['mag_v']
    return u_diff


def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    df_obs_h_data = {
        "x": u_true['lon'], "y": u_true['lat'],
        "east_velocity":  u_true['ux']*scaleunit,
        "north_velocity": u_true['uy']*scaleunit,
    }
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"]     = error_e*scaleunit
        df_obs_h_data["north_sigma"]    = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0
    df_obs_h = pd.DataFrame(data=df_obs_h_data)

    df_obs_v_data = {
        "x": u_true['lon'], "y": u_true['lat'],
        "east_velocity":  0.0,
        "north_velocity": u_true['uz']*scaleunit,
    }
    if error_v is not None:
        df_obs_v_data["east_sigma"]     = error_v*scaleunit
        df_obs_v_data["north_sigma"]    = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    return df_obs_h, df_obs_v


def build_disp_vector_grid(u_true, scaleunit, inc):
    unique_lons = u_true['lon'].unique()
    unique_lats = u_true['lat'].unique()
    n_lon = len(unique_lons); n_lat = len(unique_lats)
    print(f"Detected grid: {n_lat} x {n_lon}")
    ux_2d = u_true['ux'].to_numpy().reshape(n_lat, n_lon)
    uy_2d = u_true['uy'].to_numpy().reshape(n_lat, n_lon)
    lon_2d = u_true['lon'].to_numpy().reshape(n_lat, n_lon)
    lat_2d = u_true['lat'].to_numpy().reshape(n_lat, n_lon)
    ux_sub = ux_2d[::inc, ::inc]; uy_sub = uy_2d[::inc, ::inc]
    lon_sub = lon_2d[::inc, ::inc]; lat_sub = lat_2d[::inc, ::inc]
    ux = ux_sub.flatten(); uy = uy_sub.flatten()
    angle = np.degrees(np.arctan2(uy, ux))
    length = np.hypot(ux, uy)
    print(f"Downsampled to: {ux_sub.shape[0]} x {ux_sub.shape[1]} = {len(ux)} points")
    df_obs_h_velo = pd.DataFrame(data={
        "x": lon_sub.flatten(), "y": lat_sub.flatten(),
        "east_velocity":  ux * scaleunit,
        "north_velocity": uy * scaleunit,
    })
    df_obs_h_plot = pd.DataFrame({
        'x': lon_sub.flatten(), 'y': lat_sub.flatten(),
        'angle': angle, 'length': length * scaleunit,
    })
    return df_obs_h_velo, df_obs_h_plot


In [ ]:
# read in the observation disp. for various structure models, HOMgeneous model
u_true_hom = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, homo_str)
# build observation disp. vectors for various structure models, HOMgeneous model
df_obs_h_hom, df_obs_v_hom = build_disp_vector(u_true_hom, m2mm, error_e, error_n, error_v)
# read in the ground-turth moment, mag, and potency, HOMgeneous model
mom_true_hom = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, homo_str)

# Slab/Wedge model
u_true_sw = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, sw_str)
df_obs_h_sw, df_obs_v_sw = build_disp_vector(u_true_sw, m2mm, error_e, error_n, error_v)
mom_true_sw = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, sw_str)
# build disp difference between various structure models and homogeneous
u_true_swdiff = build_diff_disp(u_true_hom, u_true_sw)
# build differential observation disp. vectors for various structure models wrt homogeneous
df_obs_h_swdiff, df_obs_v_swdiff = build_disp_vector(u_true_swdiff, m2mm, error_e, error_n, error_v)

### OLD, 3D DeShon model, value multiplied by 4, relative to a reference
# NEW as of 03/05/2026, 3D DeShon model, value multiplied by 2.5, relative to a reference 1D 
u_true_3d = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str)
df_obs_h_3d, df_obs_v_3d = build_disp_vector(u_true_3d, m2mm, error_e, error_n, error_v)
mom_true_3d = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str)

u_true_3ddiff = build_diff_disp(u_true_hom, u_true_3d)
df_obs_h_3ddiff, df_obs_v_3ddiff = build_disp_vector(u_true_3ddiff, m2mm, error_e, error_n, error_v)

# # unamplified, original 3D DeShon model
# u_true_3d_ori = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str_ori)
# df_obs_h_3d_ori, df_obs_v_3d_ori = build_disp_vector(u_true_3d_ori, m2mm, error_e, error_n, error_v)
# mom_true_3d_ori = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str_ori)
# u_true_3ddiff_ori = build_diff_disp(u_true_hom, u_true_3d_ori)
# df_obs_h_3ddiff_ori, df_obs_v_3ddiff_ori = build_disp_vector(u_true_3ddiff_ori, m2mm, error_e, error_n, error_v)

In [ ]:
### For testing, the GT potency for diff structure models should be the same
print(mom_true_hom['potency'].to_numpy())
print(mom_true_sw['potency'].to_numpy())
print(mom_true_3d['potency'].to_numpy())
# print(mom_true_3d_ori['potency'].to_numpy())


In [ ]:
def station_grid(regiongrid, grid_spacing_deg, depth_levels):
    lon_min, lon_max = regiongrid[0], regiongrid[1]  # degrees longitude
    lat_min, lat_max = regiongrid[2], regiongrid[3]    # degrees latitude

    # Create regular lat/lon meshgrid
    lon_grid = np.arange(lon_min, lon_max + grid_spacing_deg, grid_spacing_deg)
    lat_grid = np.arange(lat_min, lat_max + grid_spacing_deg, grid_spacing_deg)
    LON_GRID, LAT_GRID = np.meshgrid(lon_grid, lat_grid)

    # Flatten for processing
    lon_2d = LON_GRID.flatten()
    lat_2d = LAT_GRID.flatten()

    # Convert 2D grid to relative coordinates (non-CK: azi_equidist returns km)
    x_2d, y_2d = utp.azi_equidist_proj(lon_2d, lat_2d, lon0, lat0)

    # Replicate the 2D grid for each depth level
    n_2d = len(x_2d)
    n_depths = len(depth_levels)
    n_total = n_2d * n_depths

    lon_dense = np.tile(lon_2d, n_depths)
    lat_dense = np.tile(lat_2d, n_depths)
    x_dense = np.tile(x_2d, n_depths)
    y_dense = np.tile(y_2d, n_depths)
    z_dense = np.repeat(depth_levels, n_2d)  # repeat each depth n_2d times

    # Create dense grid dataframe
    sta_grid = pd.DataFrame({
        'lon': lon_dense,
        'lat': lat_dense,
        'x': x_dense,
        'y': y_dense,
        'z': z_dense
    })

    return sta_grid

# Define regular lat/lon grid covering study area
# regiongrid=[-88, -83, 7.5, 12.5] 
regiongrid=[-88, -83, 7.4, 12.6] 
# regiongrid=[-87, -84, 8.6, 11.6]

# lon_min, lon_max = region[0], region[1]  # degrees longitude
# lat_min, lat_max = region[2], region[3]    # degrees latitude

# Grid resolution - adjust as needed for your visualization requirements
grid_spacing_deg = 0.01  # ~2 km spacing at this latitude
# grid_spacing_deg = 0.1  # ~20 km spacing at this latitude

# Define depth levels (0 to -80 km with 10 km increment)
# depth_levels = np.arange(0, -80 - 10, -10)  # [0, -10, -20, ..., -80]
depth_levels = [0]
print(f"Depth levels: {depth_levels} km")

# define the station_grid directly in case of inconsistency
sta_grid = station_grid(regiongrid, grid_spacing_deg, depth_levels)    
print(sta_grid.head())

## read the synthetic ground disp. at regular grid points 
# read the locations of the dense grid 
# sta_grid = read_station_grid(datadir, resultpath, meshname)


In [ ]:
# read disp at each grid point
u_true_hom_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, homo_str, grid_spacing_deg)
print(u_true_hom_grid.head())
print(u_true_hom_grid.shape)
print(u_true_hom_grid['uz'].min()*m2mm, u_true_hom_grid['uz'].max()*m2mm )
print(u_true_hom_grid['mag_h'].min()*m2mm, u_true_hom_grid['mag_h'].max()*m2mm )

# build observation disp. vectors at each grid point, downsampled
inc = 20    #increment, or downsample rate
df_obs_h_hom_grid, _ = build_disp_vector_grid(u_true_hom_grid, m2mm, inc)
# print(df_obs_h_hom_grid[:5])


In [ ]:
homo_str

In [ ]:
# Slab/Wedge model
u_true_sw_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, sw_str, grid_spacing_deg)
df_obs_h_sw_grid, _ = build_disp_vector_grid(u_true_sw_grid, m2mm, inc)
u_true_swdiff_grid = build_diff_disp(u_true_hom_grid, u_true_sw_grid)
print(u_true_swdiff_grid.head())
df_obs_h_swdiff_grid, _ = build_disp_vector_grid(u_true_swdiff_grid, m2mm, inc)

### OLD, 3D DeShon model, value multiplied by 4, relative to a reference
# NEW as of 03/05/2026, 3D DeShon model, value multiplied by 2.5, relative to a reference 1D 
u_true_3d_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, het3d_str, grid_spacing_deg)
df_obs_h_3d_grid, _ = build_disp_vector_grid(u_true_3d_grid, m2mm, inc)
u_true_3ddiff_grid = build_diff_disp(u_true_hom_grid, u_true_3d_grid)
df_obs_h_3ddiff_grid, _ = build_disp_vector_grid(u_true_3ddiff_grid, m2mm, inc)

# # unamplified, original 3D DeShon model
# u_true_3d_grid_ori = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, het3d_str_ori, grid_spacing_deg)
# df_obs_h_3d_grid_ori, _ = build_disp_vector_grid(u_true_3d_grid_ori, m2mm, inc)
# u_true_3ddiff_grid_ori = build_diff_disp(u_true_hom_grid, u_true_3d_grid_ori)
# df_obs_h_3ddiff_grid_ori, _ = build_disp_vector_grid(u_true_3ddiff_grid_ori, m2mm, inc)

In [ ]:
def plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_grid, u_true_grid, region, struc_str_for, 
                             uhcont_step=10,
                             flag_savefig=False, df_obs_h_noerr=None):

    slipsybsz = "c0.1c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)
            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*m2mm, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            
            potency = mom_true_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])
                fig.colorbar(frame=["af", "x+l@%1%S@%% magnitude", "y+l(mm)"], position="JBC+h+o0/0.5c")

            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.text(text="Stations", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic disp."])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uv_max = u_true_grid['uz'].abs().max()*m2mm     # in mm
            print('max vertical:', uv_max)
            # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
            # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
            maxdisp = np.ceil(uv_max)
            # maxdisp = 36
            print('used max vertical:', maxdisp)
            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, maxdisp/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #
                # fig.colorbar(frame=["af", "x+lU@-z@-", "y+l(mm)"]) #
                fig.colorbar(frame=["af", "x+l@%1%U@%%@-z@-", "y+l(mm)"], position="JBC+h+o0/0.5c")

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)
            # if struc_str_for == homo_str:
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="0.5p,dimgray", style="f0.4i/0.05i+l+t", fill="dimgray")
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")

            # else:
            #     # plot the contour lines of horizontal displacement vector magnitude
            #     griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            #     # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2
            #     fig.grdcontour(grid=griduh, levels=uhcont_step, annotation="+f4p, white", pen="0.25p, white") #, 2

            # scale_vec = 0.05    # 1 mm would be scaled by 'scale_vec'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # direct comparison with error-free displacement vectors at real station locations 
        if df_obs_h_noerr is not None:
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tCompare at real stations"])
                #plot the horizontal displacement vectors over the grid, error-free
                fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                        vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                #plot the horizontal displacement vectors over real station, error-free
                fig.velo(data=df_obs_h_noerr, pen="0.5p,magenta", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                    vector="0.1c+a45+p0.1p,magenta+ea+gmagenta+h0") 
                # plot legends
                fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.8", fill="white", pen=None, transparency=30, )
                lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
                fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")                

    fig.show()

    if flag_savefig: 
        figname = meshname + struc_str_for + str(slipmodel) + "_truedisp.pdf"
        fig.savefig(figpath + figname) #, crop=False
            
uh_max = u_true_hom_grid['mag_h'].max()*m2mm
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = 5*n    # vector length are plotted relative to scale_vec*length in original unit, here mm 
if slipmodel==2:
    scale_vec = 45    # vector length are plotted relative to scale_vec*length in original unit, here mm 
elif slipmodel==3:
    scale_vec = 15
elif slipmodel==4:
    scale_vec = 20    
print('max horizontal:', uh_max, 'scale vector:', scale_vec)

# #plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models, compare them at real stations without error 
# plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_hom_grid, u_true_hom_grid, region, homo_str, flag_savefig=False, df_obs_h_noerr=df_obs_h_hom1)

#plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models
plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_hom_grid, u_true_hom_grid, 
                         region1, homo_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_sw_grid, u_true_sw_grid, 
                         region1, sw_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_3d_grid, u_true_3d_grid, 
                         region1, het3d_str, flag_savefig=False)


In [ ]:
def plot_true_slip_diff_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_grid, u_true_grid, region, struc_str_for, 
                             uhcont_step=10, uvcolormax=None,
                             flag_savefig=False, df_obs_h_noerr=None):

    slipsybsz = "c0.1c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*m2mm, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

            potency = mom_true_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])
                fig.colorbar(frame=["af", "x+l@%1%S@%% magnitude", "y+l(mm)"], position="JBC+h+o0/0.5c")

            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style="s0.15c", fill="cyan", pen="0.25p,black")
            fig.text(text="Stations", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tChange in synthetic disp."])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uv_max = u_true_grid['uz'].abs().max()*m2mm     # in mm
            print('max vertical:', uv_max)
            if uvcolormax is None:
                # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
                # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
                maxdisp = np.ceil(uv_max)   
            else:
                maxdisp = uvcolormax  

            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, maxdisp/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #
                fig.colorbar(frame=["af", "x+l@~D@~@%1%U@%%@-z@-", "y+l(mm)"], position="JBC+h+o0/0.5c")

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)
            # if struc_str_for == sw_str:
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="0.5p,dimgray", style="f0.4i/0.05i+l+t", fill="dimgray")
            
            # else:
            # plot the contour lines of horizontal displacement vector magnitude
            griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2
            # fig.grdcontour(grid=griduh, levels=uhcont_step, annotation="+f4p, white", pen="0.25p, white") #, 2

            # scale_vec = 0.05    # 1 mm would be scaled by 'scale_vec'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # direct comparison with error-free displacement vectors at real station locations 
        if df_obs_h_noerr is not None:
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tCompare at real stations"])
                #plot the horizontal displacement vectors over the grid, error-free
                fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                        vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                #plot the horizontal displacement vectors over real station, error-free
                fig.velo(data=df_obs_h_noerr, pen="0.5p,magenta", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                    vector="0.1c+a45+p0.1p,magenta+ea+gmagenta+h0") 
                # plot legends
                fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.8", fill="white", pen=None, transparency=30, )
                lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
                fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")                

    fig.show()

    if flag_savefig: 
        figname = meshname + struc_str_for + str(slipmodel) + "_truedisp_diff.pdf"
        fig.savefig(figpath + figname) #, crop=False
            
#plot the differential disp vectors between models over a regular grid
print('max horizontal diff (SW):', u_true_swdiff_grid['mag_h'].max()*m2mm)
print('max horizontal diff (3D):', u_true_3ddiff_grid['mag_h'].max()*m2mm)
# print(u_true_3ddiff_grid_ori['mag_h'].max()*m2mm)

if slipmodel==2:
    # scale_vec = 15    # vector length are plotted relative to scale_vec*length in original unit, here mm 
    # uvcolormax=10
    scale_vec = 10    # NEW 3D model, x2.5, vector length are plotted relative to scale_vec*length in original unit, here mm 
    uvcolormax= 5
elif slipmodel==3:
    scale_vec = 8
    uvcolormax=6
elif slipmodel==4:
    scale_vec = 10
    uvcolormax=7    
plot_true_slip_diff_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_swdiff_grid, u_true_swdiff_grid, 
                              region1, sw_str, uhcont_step=4, uvcolormax=uvcolormax, flag_savefig=False)

scale_vec = 5    # NEW 3D model, x2.5, vector length are plotted relative to scale_vec*length in original unit, here mm 
uvcolormax= 5

plot_true_slip_diff_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_3ddiff_grid, u_true_3ddiff_grid, 
                              region1, het3d_str, uhcont_step=4, uvcolormax=uvcolormax, flag_savefig=False)

# plot_true_slip_diff_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_3ddiff_grid_ori, u_true_3ddiff_grid_ori, region1, het3d_str_ori, 
#                          uhcont_step=4, uvcolormax=uvcolormax, flag_savefig=False)                         

### Plot the summary figure for the synthetic ground displacement grid under Hom, and difference under other mu models

In [ ]:
def plot_true_slip_disp_grid_summary(mtrue_s, col2plt, scale_vec, scale_vec_diff, region, df_obs_h_hom_grid, u_true_hom_grid,                                      
                                     df_obs_h_swdiff_grid, u_true_swdiff_grid, df_obs_h_3ddiff_grid, u_true_3ddiff_grid, 
                                     uhcont_step=10, uhcont_step_diff=10, uvcolormax_diff=None, flag_savefig=False):

    slipsybsz = "c0.125c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.8c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        mu_model_labels = ["Prescribed slip (@%1%s@%%) model", "H", "S - H", "3D - H"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            step = maxslip/10
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, step], background=True, reverse=True)
            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*m2mm, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            
            potency = mom_true_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="dodgerblue", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])
                # fig.colorbar(frame=["af", "x+l@%1%s@%% magnitude", "y+l(mm)"], position="JBC+h+o0/0.5c")
                fig.colorbar(frame=[f"a{step}f{step}", "x+lSlip (@%1%s@%%) magnitude", "y+l(mm)"], position="JBC+h+o0/0.5c")

            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style="s0.15c", fill="dodgerblue", pen="0.25p,black")
            fig.text(text="Stations", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

            # Legend: net plate-motion vector V_plate at azimuth_deg (CW from N).
            # Drawn with the SAME scale and arrow style as the slip vectors above so
            # users can compare slip magnitude/orientation against the plate convergence.
            az_rad = np.deg2rad(slip_azimuth_deg)

            lgd = pd.DataFrame({
                'x': [region[0]+0.5], 'y': [region[-2]+0.35],
                'east_velocity':  [amp * np.sin(az_rad)],
                'north_velocity': [amp * np.cos(az_rad)],
            })
            fig.velo(data=lgd, pen="2p,black", spec=f"e{0.75/amp}/0",
                    vector="0.25c+a45+p1p,black+ea+gblack+h0")
            fig.text(text=f"V@-plate@-=",
                    x=region[0]+0.3, y=region[-2]+0.65,
                    font="6p,Helvetica,black", justify="MC")
            fig.text(text=f"{amp * m2mm:.0f} mm/yr",
                    x=region[0]+0.3, y=region[-2]+0.5,
                    font="6p,Helvetica,black", justify="MC")
            
            #plot panel label
            fig.text(text=panel_labels[0], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )

        title_str_prefix = "Synthetic displacement (@%1%u@%%); " 
        # title_str_prefix = ""
        color_label = "Vertical displacement (@%1%u@%%@-z@-)"

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str_prefix}{mu_model_labels[1]}"])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_hom_grid['lon'], y=u_true_hom_grid['lat'], z=u_true_hom_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uv_max = u_true_hom_grid['uz'].abs().max()*m2mm     # in mm
            # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
            # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
            maxdisp = np.ceil(uv_max)
            maxdisp = 35
            print('max vertical:', uv_max, 'used max vertical:', maxdisp)
            step = maxdisp/10
            # step = maxdisp/8
            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #
                # fig.colorbar(frame=["af", "x+lU@-z@-", "y+l(mm)"]) #
                # fig.colorbar(frame=["af", f"x+l{color_label}", "y+l(mm)"], position="JBC+h+o0/0.5c")
                fig.colorbar(frame=["a10f5", f"x+l{color_label}", "y+l(mm)"], position="JBC+h+o0/0.5c")
                # fig.colorbar(frame=[f"a{maxdisp/10*2}f{maxdisp/10}", f"x+l{color_label}", "y+l(mm)"], position="JBC+h+o0/0.5c")

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)
            # if struc_str_for == homo_str:
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.05i+l+t", fill="dimgray")

            # else:
            #     # plot the contour lines of horizontal displacement vector magnitude
            #     griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            #     # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2
            #     fig.grdcontour(grid=griduh, levels=uhcont_step, annotation="+f4p, white", pen="0.25p, white") #, 2

            # scale_vec = 0.05    # 1 mm would be scaled by 'scale_vec'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_hom_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

            #plot panel label
            fig.text(text=panel_labels[1], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )


        def plot_diff_disp_grid_panel(fig, df_obs_h_diff_grid, u_true_diff_grid, title_str, panel_label, color_label, colormap, uvcolormax=None):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str_prefix_diff}{title_str}"])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_diff_grid['lon'], y=u_true_diff_grid['lat'], z=u_true_diff_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uv_max = u_true_diff_grid['uz'].abs().max()*m2mm     # in mm
            if uvcolormax is None:
                # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
                # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
                maxdisp = np.ceil(uv_max)   
            else:
                maxdisp = uvcolormax
            print('max vertical:', uv_max, 'used max vertical:', maxdisp)

            step = maxdisp/10
            pygmt.makecpt(cmap=colormap, series=[-maxdisp, maxdisp, step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #
                fig.colorbar(frame=["af", f"x+l{color_label}", "y+l(mm)"], position="JBC+h+o0/0.5c")

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)
            # if struc_str_for == sw_str:
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.05i+l+t", fill="dimgray")
            
            # else:
            # plot the contour lines of horizontal displacement vector magnitude
            griduh = pygmt.xyz2grd(x=u_true_diff_grid['lon'], y=u_true_diff_grid['lat'], z=u_true_diff_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2
            # fig.grdcontour(grid=griduh, levels=uhcont_step_diff, annotation="+f4p, white", pen="0.25p, white") #, 2

            # scale_vec_diff = 0.05    # 1 mm would be scaled by 'scale_vec_diff'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec_diff],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec_diff), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_diff_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec_diff)+"/0.39", 
                        vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec_diff))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

            #plot panel label
            fig.text(text=panel_label, x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM")


        ### plot differential Synthetic horizontal displacement under different mu models in different panels
        colormap = "roma"
        color_label_diff = "Vertical displacement difference (@~D@~@%1%u@%%@-z@-)"
        title_str_prefix_diff = "Displacement difference (@~D@~@%1%u@%%); " 
        # title_str_prefix = ""

        # differential Synthetic horizontal displacement, SW-Hom
        with fig.set_panel(panel=[1, 0]):
            plot_diff_disp_grid_panel(fig, df_obs_h_swdiff_grid, u_true_swdiff_grid, mu_model_labels[2], panel_labels[2], 
                                      color_label_diff, colormap, uvcolormax=uvcolormax_diff)

        # differential Synthetic horizontal displacement, 3D-Hom
        with fig.set_panel(panel=[1, 1]):
            plot_diff_disp_grid_panel(fig, df_obs_h_3ddiff_grid, u_true_3ddiff_grid, mu_model_labels[3], panel_labels[3], 
                                      color_label_diff, colormap, uvcolormax=uvcolormax_diff)

    fig.show()    

    if flag_savefig: 
        # figname = meshname + "_slip" + str(slipmodel) + "_truedisp_and_diff.pdf"
        figname = meshname + inv_scheme + "_slip" + str(slipmodel) + "_ref1D_truedisp_and_diff.pdf"
        fig.savefig(figpath + figname) #, crop=False
        print(figpath + figname)


uh_max = u_true_hom_grid['mag_h'].max()*m2mm
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = 5*n    # vector length are plotted relative to scale_vec*length in original unit, here mm 
if slipmodel==2:
    scale_vec = 50    # vector length are plotted relative to scale_vec*length in original unit, here mm 
elif slipmodel==3:
    scale_vec = 15
elif slipmodel==4:
    scale_vec = 20    
print('max horizontal:', uh_max, 'scale vector:', scale_vec)

#plot the differential disp vectors between models over a regular grid
print('max horizontal diff (SW):', u_true_swdiff_grid['mag_h'].max()*m2mm)
print('max horizontal diff (3D):', u_true_3ddiff_grid['mag_h'].max()*m2mm)

if slipmodel==2:
    # scale_vec_diff = 15    # vector length are plotted relative to scale_vec*length in original unit, here mm 
    # uvcolormax_diff=10
    scale_vec_diff = 8    # NEW 3D model, x2.5, vector length are plotted relative to scale_vec*length in original unit, here mm 
    uvcolormax_diff= 5
elif slipmodel==3:
    scale_vec_diff = 8
    uvcolormax_diff=6
elif slipmodel==4:
    scale_vec_diff = 10
    uvcolormax_diff=7    


plot_true_slip_disp_grid_summary(mtrue_s, col2plt, scale_vec, scale_vec_diff, 
                                 region1, df_obs_h_hom_grid, u_true_hom_grid,                                      
                                 df_obs_h_swdiff_grid, u_true_swdiff_grid, 
                                 df_obs_h_3ddiff_grid, u_true_3ddiff_grid, 
                                 uhcont_step=10, uhcont_step_diff=4, 
                                 uvcolormax_diff=uvcolormax_diff, flag_savefig=flag_savefig)


In [ ]:
def plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h, df_obs_v, region, struc_str_for, flag_savefig=False, u_true_grid=None):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*m2mm, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # fig.plot(x=-84.869, y=10.8536, style="s0.15c", fill="red", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"], position="JBC+h+o0/0.5c")

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_h'].max()*m2mm     # in mm
                # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
                # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduh, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lHoriz. disp. mag.", "y+l(mm)"], position="JBC+h+o0/0.5c") #
                # fig.grdcontour(grid=griduh, levels=5, annotation="5+f4p, black", pen="0.25p, black") #

            #Synthetic displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduz = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_v'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduz, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lVert. disp. mag.", "y+l(mm)"], position="JBC+h+o0/0.5c") # 
                # fig.grdcontour(grid=griduz, levels=5, annotation="5+f4p, black", pen="0.25p, black") #
                    
            #Synthetic displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslip.pdf")


uh_max = np.hypot(df_obs_h_hom['east_velocity'].to_numpy(), df_obs_h_hom['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# plot the true slip model, and resulting disp vectors under the respective structure models
# plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_hom, df_obs_v_hom, region, homo_str, flag_savefig=False, u_true_grid=u_true_hom_grid)
# plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_sw, df_obs_v_sw, region, sw_str, flag_savefig=False, u_true_grid=u_true_sw_grid)
# plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_3d, df_obs_v_3d, region, het3d_str, flag_savefig=False, u_true_grid=u_true_3d_grid)

plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_hom, df_obs_v_hom, region, homo_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_sw, df_obs_v_sw, region, sw_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_3d, df_obs_v_3d, region, het3d_str, flag_savefig=False)


In [ ]:
scale_vec = 5
plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_swdiff, df_obs_v_swdiff, region, sw_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h_3ddiff, df_obs_v_3ddiff, region, het3d_str, flag_savefig=False)

In [ ]:
def plot_true_slip_disp_diff(mtrue_s, col2plt, scale_vec, maxdiff, u_true_diff_grid, df_obs_diff_h, df_obs_diff_v, region, struc_str_for, flag_savefig=False):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order]*amp*m2mm, cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # grid = pygmt.xyz2grd(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            # maxdiff = u_true_diff_grid['mag_h'].abs().max()*m2mm     # in mm
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff], background=True, reverse=True)
            fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lDiff. in horiz. disp. mag.", "y+l(mm)"]) #

            #Synthetic displacement
            fig.velo(data=df_obs_diff_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # grid = pygmt.xyz2grd(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_diff_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01))
            grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            # maxdiff = u_true_diff_grid['uz'].abs().max()*m2mm     # in mm
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff], background=True, reverse=True)
            fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lDiff. in vert. disp. mag.", "y+l(mm)"]) #
                fig.colorbar(frame=["af", "x+lDiff. in vert. disp.", "y+l(mm)"]) #

            #Synthetic displacement
            fig.velo(data=df_obs_diff_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslipdiff.pdf")

# maxdiff = round(max(u_true_swdiff_grid['mag_h'].abs().max(), u_true_swdiff_grid['mag_v'].abs().max(), 
#                     u_true_3ddiff_grid['mag_h'].abs().max(), u_true_3ddiff_grid['mag_v'].abs().max()) * m2mm)
# print("Overall max diff in disp.: ", maxdiff)
# scale_vec = 5
# plot_true_slip_disp_diff(mtrue_s, 's_dip', scale_vec, maxdiff, u_true_swdiff_grid, df_obs_h_swdiff, df_obs_v_swdiff, region, sw_str, flag_savefig=False)
# plot_true_slip_disp_diff(mtrue_s, 's_dip', scale_vec, maxdiff, u_true_3ddiff_grid, df_obs_h_3ddiff, df_obs_v_3ddiff, region, het3d_str, flag_savefig=False)


In [ ]:
# Decide the weights of the horizontal, vertical components
if pollute:
    if pollute_type == 'uniform':   
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1 
        # f_h, f_v = 1, 1/2 
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/noise_std_h), int(1/noise_std_v)
        
    elif pollute_type == 'datastd':
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1
        # f_h, f_v = 1, 1/2
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/f_h), int(1/f_v)
else:
    # Decide the weights of the horizontal, vertical components
    f_h, f_v = 1, 1
    # Print the weights of the data
    print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )

    # Take the inverse for saving the name of the weights
    w_h, w_v = int(1/f_h), int(1/f_v)


In [ ]:
# Regularization weights for the cf scheme.
# rho = sqrt(gamma/delta) is the correlation length in the prior (Bayesian).
rho_s = 1e9   # ~30 km correlation length given station spacing ~20 km

# Pick a gamma at the L-curve corner. Files exist for gamma = 1.5e2 and 1.8e2 (rho=1e9).
if slipmodel == 2:
    gamma_val_H1 = 1.5e2
    # gamma_val_H1 = 1.8e2   # alternative corner
elif slipmodel == 4:
    gamma_val_H1 = 5e1
else:
    gamma_val_H1 = 1.5e2

delta_val_L2 = gamma_val_H1 / rho_s
print(f"gamma = {gamma_val_H1:.1e}; delta = {delta_val_L2:.1e}; rho_s = {rho_s:.0e}")

# Az-consistent-forward inv_str format (matches synth_*_az_consistfwd_*nicoyaCK*.py)
inv_str = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
print("inv_str:", inv_str)


In [ ]:
from pathlib import Path

# ── Predicted moment / station displacement / inferred slip ──────────────────
def read_pred_moment(datadir, resultpath, meshname, slip_str_gt,
                     struc_str_for, inv_str, struc_str_inv):
    outFileName = 'moment_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                      names=['moment', 'mw', 'potency'])
    return mom


# Non-CK: NO rot_xy (mesh frame == geographic frame).
def read_pred_disp(u_true, datadir, resultpath, meshname, slip_str_gt,
                   struc_str_for, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+',
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

    u_res = u_pred.copy()
    u_res['ux']  = u_true['ux'] - u_pred['ux']
    u_res['uy']  = u_true['uy'] - u_pred['uy']
    u_res['uz']  = u_true['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    return u_pred, u_res


# ── Recovery ratio = the new "locking ratio": rec_amp / amp ──────────────────
# Loads slip_recovery_*.txt produced by solveCoseismicInversion_AzSlip when
# true_amp_arr is passed in. Returns DataFrame with 's_strike', 's_dip', 'mag' so the
# existing plotting code (which keys off [col2plt]) works unchanged.
# Columns of slip_recovery_*.txt:
#   xf yf zf true_amp(m/yr) rec_amp(m/yr) ratio_global ratio_local
def read_inferred_slip(datadir, resultpath, meshname, slip_str_gt,
                       struc_str_for, inv_str, struc_str_inv, type):
    outFileName = ('slip_recovery_' + meshname + slip_str_gt
                   + struc_str_for + inv_str + struc_str_inv + '.txt')
    print(outFileName)
    raw = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', comment='#',
                      header=None)
    if raw.shape[1] == 7:
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio_global','ratio_local']
    elif raw.shape[1] == 6:
        raw.columns = ['xf','yf','zf','true_amp','rec_amp','ratio_local']
        raw['ratio_global'] = raw['rec_amp'] / amp
    else:
        raise ValueError(f"Unexpected slip_recovery column count: {raw.shape[1]}")

    # Strike/dip components are stored in m_s_fault_*.txt (saved alongside);
    # only used by plots that explicitly reference s_strike/s_dip. mag suffices for ratio plots.
    msf_name = ('m_s_fault_' + meshname + slip_str_gt + struc_str_for
                + inv_str + struc_str_inv + '.txt')
    msf_path = datadir + resultpath + msf_name
    if Path(msf_path).exists():
        ms = np.loadtxt(msf_path)
        s_strike = ms[:, 0] / amp
        s_dip    = ms[:, 1] / amp
    else:
        s_strike = np.zeros(len(raw)); s_dip = np.zeros(len(raw))

    m_s = pd.DataFrame({
        's_strike': s_strike,
        's_dip':    s_dip,
        'mag':      raw['ratio_global'].values,   # recovery ratio = inferred locking ratio
    })
    print(f"  recovery ratio range: [{m_s['mag'].min():.3f}, {m_s['mag'].max():.3f}]")
    return m_s


In [ ]:
### Load the predicted surface displacements and residuals for diff forward and inversion models

## hom forward, hom inversion
u_pred_homhom, u_res_homhom = read_pred_disp(u_true_hom, datadir, resultpath, meshname, slip_str_gt, \
                                             homo_str, inv_str, homo_str)
m_s_homhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                homo_str, inv_str, homo_str, type)
mom_homhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, homo_str, inv_str, homo_str)

## slab/wedge forward, hom inversion
u_pred_swhom, u_res_swhom = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, homo_str)
m_s_swhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, homo_str, type)
mom_swhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, homo_str)

## slab/wedge forward, slab/wedge inversion
u_pred_swsw, u_res_swsw = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, sw_str)
m_s_swsw = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, sw_str, type)
mom_swsw = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, sw_str)
# difference between the heterogeneous and homogeneous model
print((m_s_swsw['s_dip']-m_s_swhom['s_dip']).min(), (m_s_swsw['s_dip']-m_s_swhom['s_dip']).max())
# print((m_s_swsw['mag']-m_s_swhom['mag']).min(), (m_s_swsw['mag']-m_s_swhom['mag']).max())


## 3d forward, hom inversion
u_pred_3dhom, u_res_3dhom = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, homo_str)
m_s_3dhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, homo_str)

## amplified 3d forward, 3d inversion
u_pred_3d3d, u_res_3d3d = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, het3d_str)
m_s_3d3d = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, het3d_str)
# difference between the heterogeneous and homogeneous model
print((m_s_3d3d['s_dip']-m_s_3dhom['s_dip']).min(), (m_s_3d3d['s_dip']-m_s_3dhom['s_dip']).max())
# print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())


# ## original 3d forward, hom inversion
# u_pred_3dhom_ori, u_res_3dhom_ori = read_pred_disp(u_true_3d_ori, datadir, resultpath, meshname, slip_str_gt, \
#                                              het3d_str_ori, inv_str, homo_str)
# m_s_3dhom_ori = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
#                                 het3d_str_ori, inv_str, homo_str, type)
# mom_3dhom_ori = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str_ori, inv_str, homo_str)

# ## original 3d forward, 3d inversion
# u_pred_3d3d_ori, u_res_3d3d_ori = read_pred_disp(u_true_3d_ori, datadir, resultpath, meshname, slip_str_gt, \
#                                              het3d_str_ori, inv_str, het3d_str_ori)
# m_s_3d3d_ori = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
#                                 het3d_str_ori, inv_str, het3d_str_ori, type)
# mom_3d3d_ori = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str_ori, inv_str, het3d_str_ori)
# # difference between the heterogeneous and homogeneous model
# print((m_s_3d3d_ori['s_dip']-m_s_3dhom['s_dip']).min(), (m_s_3d3d_ori['s_dip']-m_s_3dhom['s_dip']).max())
# # print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())



In [ ]:
# ### For testing, the GT potency for diff structure models should be the same
# print(mom_homhom['potency'].to_numpy())
# print(mom_swhom['potency'].to_numpy())
# print(mom_swsw['potency'].to_numpy())
# print(mom_3dhom['potency'].to_numpy())
# print(mom_3d3d['potency'].to_numpy())

In [ ]:
# assess_col = 's_dip'   # assessment component
assess_col = 'mag'   # assessment component


In [ ]:
# # if slipmodel == 1 or slipmodel == 2 or slipmodel == 3:
# # Global assessment on the recovery
# gmetrics_homhom = utp.GlobalSlipMetrics(mtrue_s, m_s_homhom, loc_f)
# gmetrics_swsw = utp.GlobalSlipMetrics(mtrue_s, m_s_swsw, loc_f)
# gmetrics_3d3d = utp.GlobalSlipMetrics(mtrue_s, m_s_3d3d, loc_f)

# print('Global recovery assessment: ')
# recovery_thresh = 0.8   # threshold to take as recovered, in fraction

# print('-- From Homogeneous inversion -- ')
# print(gmetrics_homhom.summary(col=assess_col, threshold=recovery_thresh))

# print('-- From Slab/wedge contrast inversion -- ')
# print(gmetrics_swsw.summary(col=assess_col, threshold=recovery_thresh))

# print('-- From 3D model inversion -- ')
# print(gmetrics_3d3d.summary(col=assess_col, threshold=recovery_thresh))

In [ ]:
# if slipmodel == 1 or slipmodel == 2 or slipmodel == 3:
#     # Individual assessment on the recovery
#     metrics_homhom = utp.SlipRecoveryMetrics(mtrue_s, m_s_homhom, loc_f)
#     metrics_swsw = utp.SlipRecoveryMetrics(mtrue_s, m_s_swsw, loc_f)
#     metrics_3d3d = utp.SlipRecoveryMetrics(mtrue_s, m_s_3d3d, loc_f)

#     if slipmodel == 2:
#         ## offshore stripe
#         mask_shallow = (mtrue_s[assess_col] == 1.0) & (loc_f['zf'] > -25)
#         print('Offshore stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         ## onshore stripe
#         mask_deep = (mtrue_s[assess_col] == 1.0) & (loc_f['zf'] < -25)
#         print('Onshore stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         ## gap in between 
#         mask_gap = (mtrue_s[assess_col] < 1e-5) & (loc_f['zf'] > -50) & (loc_f['zf'] < -10)
#         print('Gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap, col=assess_col, type='gap'))
    
#     elif slipmodel == 3:
#         ## offshore gap
#         mask_gap1 = (mtrue_s[assess_col] < 1e-5) & (loc_f['zf'] > -25)
#         print('Offshore gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap1, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap1, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap1, col=assess_col, type='gap'))

#         ## onshore gap
#         mask_gap2 = (mtrue_s[assess_col] < 1e-5) & (loc_f['zf'] < -25)
#         print('Onshore gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap2, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap2, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap2, col=assess_col, type='gap'))

#         ## intermediate stripe 
#         mask_inter = (mtrue_s[assess_col] == 1.0)
#         print('Intermediate stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

In [ ]:
def plot_resolution_comparison(mtrue_s, m_s_hom, m_s_sw, m_s_3d, col2plt, region, absflag=False, flag_savefig=False):

    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=2, figsize=("10c", "12.8c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue locking model"])
            # maxslip = mtrue_s[col2plt].max()
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            if not absflag:
                order = mtrue_s[col2plt].argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order], cmap=True)
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                order = mtrue_s[col2plt].abs().argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order].abs(), cmap=True)
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt].abs(), cmap=True)

            potency = mom_true_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
           
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="dodgerblue", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lTrue locking ratio"], position="JBC+h+o0/0.5c")
            
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style="s0.15c", fill="dodgerblue", pen="0.25p,black")
            fig.text(text="Stations", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

            #plot panel label
            fig.text(text=panel_labels[0], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )

        # Slip inferred from the inversion under homogeneous model with syn from same model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tH_H"])
            # maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)    #m_s_hom['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not absflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_hom[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_homhom.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_homhom.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_homhom.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_homhom.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_homhom.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_homhom.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            potency = mom_homhom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInferred locking ratio"], position="JBC+h+o0/0.5c") #, "y+l(mm)"

            #plot panel label
            fig.text(text=panel_labels[1], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )

        # slip inferred from the inversion under the Slab/Wedge contrast model with syn from same model
        with fig.set_panel(panel=[1, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tS_S"])
            # maxslip = m_s_sw[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)    #m_s_sw['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not absflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_sw[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_swsw.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_swsw.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_swsw.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_swsw.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_swsw.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_swsw.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            potency = mom_swsw['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInferred locking ratio"], position="JBC+h+o0/0.5c")

            #plot panel label
            fig.text(text=panel_labels[2], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )

        # slip inferred from the inversion under 3D heterogeneity model with syn from same model
        with fig.set_panel(panel=[1, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D_3D"])
            # maxslip = m_s[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)    #m_s_3d['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not absflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_3d[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_3d3d.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_3d3d.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_3d3d.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_3d3d.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_3d3d.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_3d3d.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            potency = mom_3d3d['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInferred locking ratio"], position="JBC+h+o0/0.5c")

            #plot panel label
            fig.text(text=panel_labels[3], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )
            
    fig.show()

    if flag_savefig:
        if not absflag:
            figname = meshname + "_slip" + str(slipmodel) + "ref1D_sum_resol_all.pdf"
        else:
            figname = meshname + "_slip" + str(slipmodel) + "ref1D_sum_resol_diff_re_true_all.pdf"
        
        fig.savefig(figpath + figname)
        print(figpath + figname)


# plot the comparison between true slip and recovery under diff. structure models
plot_resolution_comparison(mtrue_s, m_s_homhom, m_s_swsw, m_s_3d3d, assess_col, region1, absflag=False, flag_savefig=flag_savefig)
# plot_resolution_comparison(mtrue_s, m_s_homhom-mtrue_s, m_s_swsw-mtrue_s, m_s_3d3d-mtrue_s, assess_col, region1, absflag=True, flag_savefig=True)

# plot_resolution_comparison(mtrue_s, m_s_swsw, m_s_swsw, m_s_swsw, assess_col, region, flag_savefig=True)
# plot_resolution_comparison(mtrue_s, m_s_homhom, m_s_homhom, m_s_homhom, assess_col, region, flag_savefig=True)


In [ ]:
def plot_resolution_comparison2(m_s_hom, m_s_sw, m_s_3d, col2plt, region, diffflag=False, flag_savefig=False):

    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        if not diffflag:
            titlestr = ""
        else:
            titlestr = " - True"

        # Slip inferred from the inversion under homogeneous model with syn from same model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+tH_H{titlestr}"])
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            nstep = 10
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/nstep], background=True, reverse=True)    #m_s_hom['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not diffflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_hom[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_homhom.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_homhom.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_homhom.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_homhom.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_homhom.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_homhom.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")

            if not diffflag:
                potency = mom_homhom['potency'].to_numpy().item()
                fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            if not diffflag:
                colorlabel = "Inferred locking ratio"
            else:
                colorlabel = "Absolute Difference in locking ratio"
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"


        # slip inferred from the inversion under the Slab/Wedge contrast model with syn from same model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+tSW_SW{titlestr}"])
            # maxslip = m_s_sw[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/nstep], background=True, reverse=True)    #m_s_sw['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not diffflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_sw[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_swsw.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_swsw.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_swsw.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_swsw.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_swsw.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_swsw.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_sw[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            if not diffflag:
                potency = mom_swsw['potency'].to_numpy().item()
                fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")


        # slip inferred from the inversion under 3D heterogeneity model with syn from same model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t3D_3D{titlestr}"])
            # maxslip = m_s[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/nstep], background=True, reverse=True)    #m_s_3d['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not diffflag:
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_3d[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_3d3d.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_3d3d.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_3d3d.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_3d3d.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_3d3d.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_3d3d.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            if not diffflag:
                potency = mom_3d3d['potency'].to_numpy().item()
                fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")

    fig.show()

    if flag_savefig:
        if not diffflag:
            fig.savefig(figpath + meshname + "_slip" + str(slipmodel) + "_sum_invslip.pdf")
        else:
            fig.savefig(figpath + meshname + "_slip" + str(slipmodel) + "_sum_invslip_diff_re_true.pdf")


# plot the comparison between true slip and recovery under diff. structure models
plot_resolution_comparison2(m_s_homhom, m_s_swsw, m_s_3d3d, assess_col, region1, diffflag=False, flag_savefig=flag_savefig)
# plot_resolution_comparison2(m_s_homhom-mtrue_s, m_s_swsw-mtrue_s, m_s_3d3d-mtrue_s, assess_col, region1, diffflag=True, flag_savefig=True)


In [ ]:
def plot_sw_3d_hom_slip_diff(m_s_hom, m_s_sw, m_s_3d, col2plt, region, ncols=3, maxdiff_plot=None, flag_savefig=False):
    
    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"
    # colormap = "rainbow"

    figheight = "10c"
    figwidth = f"{5*ncols}c"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=ncols, figsize=(figwidth, figheight), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # mu_model_labels = ["S_S - S_H", "3D_3D - 3D_H", "S_S - S_H", "3D_3D - 3D_H"]
        mu_model_labels = ["S - H", "3D - H", "S - H", "3D - H"]
        # plate_labels = ["Rough", "Smooth"]
        plate_labels = ["Irregular", "Regular"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        if ncols == 3:
            # Slip inferred from the inversion under homogeneous model with syn from same model
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+tH"])
                maxslip = 1
                # maxslip = mtrue_s[col2plt].max()
                # maxslip = m_s_hom[col2plt].max()
                # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
                pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom['mag'].max()
                # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
                # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
                # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

                order = m_s_hom[col2plt].argsort()
                fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_hom[col2plt].iloc[order], cmap=True)
                # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation

                # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
                # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
                potency = mom_homhom['potency'].to_numpy().item()
                fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

                scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
                mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
                fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
                # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
                fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
                fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
                # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

                colorlabel = "Inferred locking ratio"
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"

                #plot panel label
                fig.text(text=panel_labels[2], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
                fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                        fill="black@50")
                    
        # difference between the SW and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            print((m_s_sw[col2plt] - m_s_hom[col2plt]).min(), (m_s_sw[col2plt] - m_s_hom[col2plt]).max())
            maxdiff = np.max([(m_s_sw[col2plt] - m_s_hom[col2plt]).abs().max(), (m_s_3d[col2plt] - m_s_hom[col2plt]).abs().max()])
            maxdiff = (m_s_sw[col2plt] - m_s_hom[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            diff_step = maxdiff / 10    
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = 0.5
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_plot = m_s_sw[col2plt] - m_s_hom[col2plt]
            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_sw[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_hom[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_plot.min():.2f}, {m_s_plot.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)            
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            # colorlabel = "Difference in locking ratio"
            colorlabel = "Difference in locking ratio"
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[0], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")


        # ABS difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[1]}"])
            print((m_s_3d[col2plt] - m_s_hom[col2plt]).min(), (m_s_3d[col2plt] - m_s_hom[col2plt]).max())
            maxdiff = (m_s_3d[col2plt] - m_s_hom[col2plt]).abs().max()
            print(maxdiff)
            if maxdiff_plot is not None:
                maxdiff = maxdiff_plot
            diff_step = maxdiff / 10    
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, diff_step], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)        
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            m_s_plot = m_s_3d[col2plt] - m_s_hom[col2plt]
            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_3d[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{m_s_plot.min():.2f}, {m_s_plot.max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
        
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            colorlabel = "Difference in locking ratio"
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
                # fig.colorbar(frame=[f"a{diff_step*5}f{diff_step}", f"x+l{colorlabel}"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[1], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")
                
    fig.show()

    if flag_savefig:
        # figname = meshname + "_slip" + str(slipmodel) + "_sum_invslip_diff_re_hom.pdf"
        figname = meshname + "_slip" + str(slipmodel) + "ref1D_sum_invslip_diff_re_hom.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

    # return fig

## difference relative to the homo model
if slipmodel == 2:
    # maxdiff_plot = 0.18
    # maxdiff_plot = 0.08     # NEW 3D model, x2.5 relative to 1D reference
    maxdiff_plot = 0.05     # NEW 3D model, x2.5 relative to 1D reference
elif slipmodel == 4:
    # maxdiff_plot = 0.12
    maxdiff_plot = 0.05    # NEW 3D model, x2.5 relative to 1D reference

print('recovery ratio (H):', f"{m_s_homhom[assess_col].min():.3f}, {m_s_homhom[assess_col].max():.3f}")
print('recovery ratio (SW):', f"{m_s_swsw[assess_col].min():.3f}, {m_s_swsw[assess_col].max():.3f}")
print('recovery ratio (3D):', f"{m_s_3d3d[assess_col].min():.3f}, {m_s_3d3d[assess_col].max():.3f}")

# plot_sw_3d_hom_slip_diff(m_s_homhom, m_s_swsw, m_s_3d3d, assess_col, region1, maxdiff_plot, flag_savefig=True)
plot_sw_3d_hom_slip_diff(m_s_homhom, m_s_swsw, m_s_3d3d, assess_col, region1, ncols=2, maxdiff_plot=maxdiff_plot, flag_savefig=True)
# plot_sw_3d_hom_slip_diff(m_s_homhom, m_s_swsw, m_s_3d3d, assess_col, region1, maxdiff_plot=None, flag_savefig=False)

# # for testing
# plot_sw_3d_hom_slip_diff(m_s_homhom, m_s_swsw, m_s_3d3d_ori, assess_col, region1, ncols=2, maxdiff_plot=maxdiff_plot, flag_savefig=False)


In [ ]:
# # recovered slip along thin profiles
# loc_f['xf'], loc_f['yf'] = rot_xy(loc_f['xf'], loc_f['yf'], -rot) 

# # Option 1: Thin slice (10 km thickness)
# bin_width_km = 10
# thickness_km = 10
# prof1_true = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=mtrue_s[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_true = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=mtrue_s[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_true = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=mtrue_s[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# prof1_hom = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_homhom[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_hom = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_homhom[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_hom = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_homhom[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# prof1_sw = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_swsw[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_sw = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_swsw[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_sw = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_swsw[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# prof1_3d = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_3d = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_3d = utp.extract_directional_profile(x_coords=loc_f['xf'], y_coords=loc_f['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# profile_sets = [
#     [prof1_hom, prof1_sw, prof1_3d, prof1_true],  # Location A
#     [prof2_hom, prof2_sw, prof2_3d, prof2_true],  # Location B  
#     [prof3_hom, prof3_sw, prof3_3d, prof3_true]   # Location C
# ]
# utp.plot_profile_enhanced(profile_sets, loc_f['xf'].values, loc_f['yf'].values,
#                       m_s_homhom[assess_col].values, m_s_swsw[assess_col].values,
#                       m_s_3d3d[assess_col].values, mtrue_s[assess_col].values)

In [ ]:
def plot_homhetvstrue_slip(m_s_hom, m_s, mtrue_s, col2plt, region, struc_str_for, isreftrue=True, flag_savefig=False):

    slipsybsz = "c0.1c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the hom-true slip vs. het-true slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # True Slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tRef slip"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tRef slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"], position="JBC+h+o0/0.5c")
        
        # ABS difference between the homogeneous and true model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom - Ref"])
            maxdiff = np.max([(m_s_hom[col2plt] - mtrue_s[col2plt]).abs().max(), (m_s[col2plt] - mtrue_s[col2plt]).abs().max()])
            print(maxdiff)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = 0.5
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt] - mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_hom[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)            
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            #if the reference slip is true slip, show the rmse
            if isreftrue:
                aa = (m_s_hom[col2plt] - mtrue_s[col2plt]).abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # metrics_hom = utp.SlipRecoveryMetrics(mtrue_s, m_s_hom, loc_f)
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_hom.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_hom.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_hom.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_hom.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_hom.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_hom.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference"], position="JBC+h+o0/0.5c")

        # ABS difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet - Ref"])
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)        
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            #if the reference slip is true slip, show the rmse
            if isreftrue:
                aa = (m_s[col2plt] - mtrue_s[col2plt]).abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="6p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="6p,Helvetica,black", justify="ML")
                # metrics_het = utp.SlipRecoveryMetrics(mtrue_s, m_s, loc_f)
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_het.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_het.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_het.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_het.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_het.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_het.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="6p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"], position="JBC+h+o0/0.5c")
                fig.colorbar(frame=["af", "x+lDifference"], position="JBC+h+o0/0.5c")
                
    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_slipdiff.pdf")

## difference relative to the true model
# plot_homhetvstrue_slip(m_s_homhom, m_s_homhom, mtrue_s, assess_col, region, homo_str, flag_savefig=False)    
# plot_homhetvstrue_slip(m_s_swsw, m_s_3d3d, mtrue_s, assess_col, region, homo_str, flag_savefig=False)    

## difference relative to the homo model
plot_homhetvstrue_slip(m_s_swsw, m_s_3d3d, m_s_homhom, assess_col, region, homo_str, isreftrue=False, flag_savefig=False)  

# Under the same observation (displacements) from a heterogeneous structure, how does inversion vary if ignoring the structure?

* In other words, the inversion is based on the same synthetic data, which is generated from the respective heterogeneous structure (either Slab/Wedge model or amplified 3D model)


In [ ]:
# Pick the gamma to use for the het-forward / hom-inversion comparison plots below.
# By default: same gamma as cell 25 (so cells 38-46 reuse the data already loaded in cell 27).
# To use a DIFFERENT gamma for this specific experiment, override here.
if slipmodel == 2:
    gamma_val_H1 = 1.5e2
    # gamma_val_H1 = 1.8e2   # alternative corner gamma
elif slipmodel == 4:
    gamma_val_H1 = 5e1
else:
    gamma_val_H1 = 1.5e2

delta_val_L2 = gamma_val_H1 / rho_s
print(f"gamma = {gamma_val_H1:.1e}; delta = {delta_val_L2:.1e}")

# Az-consistent-forward inv_str (must match cell 25's format)
inv_str = f"_cf_synscaamp_azbd_w{w_h}{w_v}_gs{gamma_val_H1:.1e}_ds{delta_val_L2:.1e}"
print("inv_str:", inv_str)


In [ ]:
## slab/wedge forward, hom inversion
u_pred_swhom1, u_res_swhom1 = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, homo_str)
m_s_swhom1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, homo_str, type)
mom_swhom1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, homo_str)

## slab/wedge forward, slab/wedge inversion
u_pred_swsw1, u_res_swsw1 = read_pred_disp(u_true_sw, datadir, resultpath, meshname, slip_str_gt, \
                                             sw_str, inv_str, sw_str)
m_s_swsw1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                sw_str, inv_str, sw_str, type)
mom_swsw1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, sw_str, inv_str, sw_str)
# difference between the heterogeneous and homogeneous model
print((m_s_swsw1['s_dip']-m_s_swhom1['s_dip']).min(), (m_s_swsw1['s_dip']-m_s_swhom1['s_dip']).max())
# print((m_s_swsw['mag']-m_s_swhom['mag']).min(), (m_s_swsw['mag']-m_s_swhom['mag']).max())


## 3d forward, hom inversion
u_pred_3dhom1, u_res_3dhom1 = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, homo_str)
m_s_3dhom1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, homo_str)

## 3d forward, 3d inversion
u_pred_3d3d1, u_res_3d3d1 = read_pred_disp(u_true_3d, datadir, resultpath, meshname, slip_str_gt, \
                                             het3d_str, inv_str, het3d_str)
m_s_3d3d1 = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d1 = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, het3d_str)
# difference between the heterogeneous and homogeneous model
print((m_s_3d3d1['s_dip']-m_s_3dhom1['s_dip']).min(), (m_s_3d3d1['s_dip']-m_s_3dhom1['s_dip']).max())
# print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())


# ## original 3d forward, hom inversion
# u_pred_3dhom1_ori, u_res_3dhom1_ori = read_pred_disp(u_true_3d_ori, datadir, resultpath, meshname, slip_str_gt, \
#                                              het3d_str_ori, inv_str, homo_str)
# m_s_3dhom1_ori = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
#                                 het3d_str_ori, inv_str, homo_str, type)
# mom_3dhom1_ori = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str_ori, inv_str, homo_str)

# ## original 3d forward, 3d inversion
# u_pred_3d3d1_ori, u_res_3d3d1_ori = read_pred_disp(u_true_3d_ori, datadir, resultpath, meshname, slip_str_gt, \
#                                              het3d_str_ori, inv_str, het3d_str_ori)
# m_s_3d3d1_ori = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, \
#                                 het3d_str_ori, inv_str, het3d_str_ori, type)
# mom_3d3d1_ori = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str_ori, inv_str, het3d_str_ori)
# # difference between the heterogeneous and homogeneous model
# print((m_s_3d3d1_ori['s_dip']-m_s_3dhom1['s_dip']).min(), (m_s_3d3d1_ori['s_dip']-m_s_3dhom1['s_dip']).max())
# # print((m_s_3d3d['mag']-m_s_3dhom['mag']).min(), (m_s_3d3d['mag']-m_s_3dhom['mag']).max())

In [ ]:
# assess_col = 's_dip'   # assessment component
assess_col = 'mag'   # assessment component

In [ ]:
def plot_homvshet_slip2(mtrue_s, m_s_hom, m_s, col2plt, region, struc_str_for, flag_savefig=False):

    slipsybsz = "c0.1c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        

        # difference in slip inferred from the inversion under WRONG mu model compare to true slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2"])
            # maxslip = m_s[col2plt].max()
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_3d['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation   
            
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_hom[col2plt]-mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation 

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
           
            # fig.text(text=f"@~D@~ locking: [{(m_s_hom[col2plt] - mtrue_s[col2plt]).min():.2f}, {(m_s_hom[col2plt] - mtrue_s[col2plt]).max():.2f}]",
            #          x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
                        
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lAbsolute Difference in locking ratio"], position="JBC+h+o0/0.5c")      
                
        # difference in slip inferred from the inversion under correct mu model compare to true slip
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2"])
            # maxslip = m_s[col2plt].max()
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_3d['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation   
            
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt]-mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation 

            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
           
            # fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - mtrue_s[col2plt]).min():.2f}, {(m_s[col2plt] - mtrue_s[col2plt]).max():.2f}]",
            #          x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
                        
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lAbs. Difference in locking ratio"], position="JBC+h+o0/0.5c")

        # Difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2"])
            maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
            maxdiff = 0.25
            print(maxdiff)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p, darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lLocking difference"], position="JBC+h+o0/0.5c")          

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_for + "_slip" + str(slipmodel) + "_hethom_diff_slip.pdf")

plot_homvshet_slip2(mtrue_s, m_s_swhom1, m_s_swsw1, assess_col, region1, sw_str, flag_savefig=False) 

plot_homvshet_slip2(mtrue_s, m_s_3dhom1, m_s_3d3d1, assess_col, region1, het3d_str, flag_savefig=False) 


In [ ]:
def plot_homvshet_slip_summary(m_s_swhom, m_s_swsw, m_s_3dhom, m_s_3d3d, col2plt, maxdiff, region, flag_savefig=False):

    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # mu_model_labels = ["S_S - S_H", "3D_3D - 3D_H", "S_S - S_H", "3D_3D - 3D_H"]
        mu_model_labels = ["S - H", "3D - H", "S - H", "3D - H"]
        # plate_labels = ["Rough", "Smooth"]
        plate_labels = ["Irregular", "Regular"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)"]

        # Difference between the SW and homogeneous model
        with fig.set_panel(panel=[0, 0]):
            m_s_plot = m_s_swsw[col2plt] - m_s_swhom[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.2p, darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["a0.1f0.02", "x+lLocking difference"], position="JBC+h+o0/0.5c")          

            #plot panel label
            fig.text(text=panel_labels[0], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")

        # Difference between the 3D and homogeneous model
        with fig.set_panel(panel=[0, 1]):
            m_s_plot = m_s_3d3d[col2plt] - m_s_3dhom[col2plt]
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[1]}"])
            # maxdiff = (m_s_plot).abs().max()
            # maxdiff = 0.25
            # print(maxdiff)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = m_s_plot.argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s_plot.iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s_plot).abs(), cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_plot, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s_plot).min():.2f}, {(m_s_plot).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.2p, darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["a0.1f0.02", "x+lLocking difference"], position="JBC+h+o0/0.5c")          
            
            #plot panel label
            fig.text(text=panel_labels[1], x=region[0]+0.15, y=region[-1]-0.1, font="9p,Helvetica-Bold,black", justify="CM")
            fig.text(text=plate_labels[0], x=region[1]-0.05, y=region[-1]-0.1, font="8p,Helvetica-Bold,white", justify="RM",
                     fill="black@50")

    fig.show()

    if flag_savefig:
        # figname = meshname + "_slip" + str(slipmodel) + "_hethom_diff_sum.pdf"
        figname = meshname + "_slip" + str(slipmodel) + "ref1D_hethom_diff_sum.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

# maxdiff = (m_s_plot).abs().max()

maxdiff = np.max( [(m_s_swsw1[assess_col] - m_s_swhom1[assess_col]).abs().max(), 
                (m_s_3d3d1[assess_col] - m_s_3dhom1[assess_col]).abs().max()]) 
print(maxdiff)

if slipmodel == 2:  
    maxdiff = 0.32
    # maxdiff = 0.29
    # maxdiff = 0.34
    maxdiff = 0.20      # NEW 3D model, x2.5 relative to 1D reference

elif slipmodel == 4:  
    # maxdiff = 0.40
    maxdiff = 0.20    # NEW 3D model, x2.5 relative to 1D reference

plot_homvshet_slip_summary(m_s_swhom1, m_s_swsw1, m_s_3dhom1, m_s_3d3d1, assess_col, maxdiff, region1, flag_savefig=True)        

# # for testing
# plot_homvshet_slip_summary(m_s_swhom1, m_s_swsw1, m_s_3dhom1_ori, m_s_3d3d1_ori, assess_col, maxdiff, region1, flag_savefig=False)        


In [ ]:
def plot_homvshet_slip(m_s_hom, m_s, col2plt, region, struc_str_for, mom_hom, mom, flag_savefig=False):

    slipsybsz = "c0.11c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation

            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)

            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            potency = mom_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInferred locking ratio"], position="JBC+h+o0/0.5c") #, "y+l(mm)"

        # slip inferred from the heterogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet. inv."])
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['locking'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt].abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            potency = mom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lInferred locking ratio"], position="JBC+h+o0/0.5c")

        # Difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet - Hom"])
            if struc_str_for == homo_str:
                maxdiff = 10
            else:
                maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
                maxdiff = 0.25
                print(maxdiff)
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.text(text=f"@~D@~ locking: [{(m_s[col2plt] - m_s_hom[col2plt]).min():.2f}, {(m_s[col2plt] - m_s_hom[col2plt]).max():.2f}]",
                     x=region[0]+0.1, y=region[-2]+0.1, font="6p,Helvetica,black", justify="ML")
            
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference"], position="JBC+h+o0/0.5c")

    fig.show()

    if flag_savefig:
        # print('yes')
        # if struc_str_for == homo_str:
        #     fig.savefig(figpath + meshname + "_hom_slip.pdf")
        # else:
        fig.savefig(figpath + meshname + "_slip" + str(slipmodel) + struc_str_for + "_hethomslip_diff.pdf")

    # return fig

## difference relative to the true model
# fig = plot_sw_3d_hom_slip_diff(mtrue_s, m_s_swhom1, m_s_swsw1, assess_col, region1, flag_savefig=False)
plot_homhetvstrue_slip(m_s_swhom1, m_s_swsw1, mtrue_s, assess_col, region1, homo_str, flag_savefig=False)   

## slip difference under inversion using hom or het models based on same data from het models
plot_homvshet_slip(m_s_swhom1, m_s_swsw1, assess_col, region1, sw_str, mom_swhom1, mom_swsw1, flag_savefig=False) 
# fig.savefig(figpath + meshname + "_slip" + str(slipmodel) + "_hethomslip_diff.pdf")

# plot_homvshet_slip(m_s_swhom, m_s_swsw, assess_col, region1, sw_str, mom_swhom, mom_swsw, flag_savefig=True)   

# same thing, but for 3D model
plot_homhetvstrue_slip(m_s_3dhom, m_s_3d3d, mtrue_s, assess_col, region1, homo_str, flag_savefig=False)  
plot_homvshet_slip(m_s_3dhom1, m_s_3d3d1, assess_col, region1, het3d_str, mom_3dhom1, mom_3d3d1, flag_savefig=False) 


### Plot a summary figure for inferred locking under all mu models

In [ ]:
def plot_resolution_summary(mtrue_s, m_s_homhom, m_s_swsw, m_s_swhom, m_s_3d3d, m_s_3dhom,
                            mom_true_hom, mom_homhom, mom_swsw, mom_swhom, mom_3d3d, mom_3dhom, 
                            col2plt, region, flag_savefig=False):

    slipsybsz = "c0.11c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=3, figsize=("15c", "12.7c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="8p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.15c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        mu_model_labels = ["True locking model", "H_H", "S_S", "3D_3D", "S_H", "3D_H"]
        panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{mu_model_labels[0]}"])
            # maxslip = mtrue_s[col2plt].max()
            maxslip = 1
            step = maxslip/10
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, step], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            order = mtrue_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=mtrue_s[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation

            potency = mom_true_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")
           
            scale_lon, scale_lat = region[1]-0.35, region[-1]-0.4
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="dodgerblue", pen="0.25p,black")

            color_label = "True locking ratio"
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{color_label}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
            
            #plot legends
            fig.plot(x=region[0]+0.3, y=region[-2]+0.2, style="r1/0.6", fill="white", pen="0.1p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.3, style="s0.15c", fill="dodgerblue", pen="0.25p,black")
            fig.text(text="Stations", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

            #plot panel label
            fig.text(text=panel_labels[0], x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM", 
                    #  pen="0.5p,black"
                     )


        # define a function to plot each panel, as they are quite similar
        def plot_resolution_panel(fig, m_s, mom, title_str, panel_label, color_label, colormap, maxslip):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", f"+t{title_str_prefix}{title_str}"])
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, step], background=True, reverse=True)    #m_s_hom['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            order = m_s[col2plt].argsort()
            fig.plot(x=loc_f["lon"].iloc[order], y=loc_f["lat"].iloc[order], style=slipsybsz, fill=m_s[col2plt].iloc[order], cmap=True)
            # fig.plot(x=loc_f['lon'], y=loc_f['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation

            # fig.grdcontour(grid=grid, levels=[0.8], annotation="0.5+f8p", pen="1p,white")
            # fig.grdcontour(grid=grid, levels=[0.9], annotation="0.5+f8p", pen="1p,black")

            potency = mom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.05, y=region[-1]-0.1, font="6p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.05, y=region[-1]-0.2, font="6p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.2p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.3p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            # colorlabel = "Inferred locking ratio"
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", f"x+l{color_label}"], position="JBC+h+o0/0.5c") #, "y+l(mm)"
        
            #plot panel label
            fig.text(text=panel_label, x=region[1]-0.15, y=region[-2]+0.2, font="9p,Helvetica-Bold,black", justify="CM")


        ### plot inferred locking under different mu models in different panels
        color_label = "Inferred locking ratio"
        # title_str_prefix = "Inferred locking " 
        title_str_prefix = ""

        # Slip inferred from the inversion under homogeneous model with syn from same model
        with fig.set_panel(panel=[0, 1]):     #[1, 0]
            plot_resolution_panel(fig, m_s_homhom, mom_homhom, mu_model_labels[1], panel_labels[1], color_label, colormap, maxslip)

        # slip inferred from the inversion under the Slab/Wedge contrast model with syn from same model
        with fig.set_panel(panel=[0, 2]):      #[0, 1]
            plot_resolution_panel(fig, m_s_swsw, mom_swsw, mu_model_labels[2], panel_labels[2], color_label, colormap, maxslip)

        # slip inferred from the inversion under 3D heterogeneity model with syn from same model
        with fig.set_panel(panel=[1, 0]):   #[1, 1]
            plot_resolution_panel(fig, m_s_3d3d, mom_3d3d, mu_model_labels[3], panel_labels[3], color_label, colormap, maxslip)

        # slip inferred from the inversion under the homogeneous model, but syn from Slab/Wedge contrast model 
        with fig.set_panel(panel=[1, 1]):   #[0, 2]
            plot_resolution_panel(fig, m_s_swhom, mom_swhom, mu_model_labels[4], panel_labels[4], color_label, colormap, maxslip)

        # slip inferred from the inversion under the homogeneous model, but syn from 3D heterogeneity model
        with fig.set_panel(panel=[1, 2]):   #[1, 2]
            plot_resolution_panel(fig, m_s_3dhom, mom_3dhom, mu_model_labels[5], panel_labels[5], color_label, colormap, maxslip)               
      
    fig.show()

    if flag_savefig:
        # figname = meshname + "_slip" + str(slipmodel) + "_sum_invslip_all.pdf"
        figname = meshname + "_slip" + str(slipmodel) + "ref1D_sum_invslip_all.pdf"
        # figname = meshname + "_slip" + str(slipmodel) + "_sum_invslip_all_1.pdf"
        fig.savefig(figpath + figname)
        print(figpath + figname)

# plot locking resolution summary, true vs. hom-hom, sw-sw, sw-hom, 3d-3d, 3d-hom
plot_resolution_summary(mtrue_s, m_s_homhom, m_s_swsw, m_s_swhom1, m_s_3d3d, m_s_3dhom1,
                        mom_true_hom, mom_homhom, mom_swsw, mom_swhom1, mom_3d3d, mom_3dhom1, 
                        assess_col, region1, flag_savefig=flag_savefig)

In [ ]:
# # plot the histogram of difference in slip
# fig, ax = plt.subplots(figsize=(4, 4.5), dpi=300)

# xmin, xmax = -maxdiff, maxdiff
# bin_width = 0.5
# bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
# median = np.median(m_s_mag['diff'])
# sigma = np.std(m_s_mag['diff'])
# # rounded_sigma = round(sigma)
# counts1, _, _ = ax.hist(m_s_mag['diff'], bins=bins, color='dimgray', alpha=0.7, edgecolor='black')
# ymin, ymax = 0, np.max(counts1) + 100
# ax.errorbar(median, ymax*0.82, xerr=sigma, fmt='o', color='black', capsize=5, capthick=2, 
#             linewidth=2, label=fr'$\pm1\sigma = \pm{sigma:.3f}$')
# ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median = {median:.3f}')
# ax.legend()
# ax.set_xlabel('Difference')
# ax.set_ylabel('Frequency')
# ax.set_title('Difference in coupling (Het-Hom)', fontweight='bold')
# ax.set_xlim(xmin, xmax)
# # ax.set_ylim(ymin, ymax)

# Data fit performance 

In [ ]:
def build_pred_disp_vector(df, u_pred, u_res, error_e, error_n, error_v):
    ### data fitting by the heterogeneous inversion 
    # convert from m to mm
    df_pred_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_pred['ux']*1000,
            "north_velocity": u_pred['uy']*1000,        
        }
    )
    df_pred_h.head()

    df_pred_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_pred['uz']*1000,        
        }
    )

    # convert from m to mm, in order to directly compare with 
    df_res_h = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": u_res['ux']*1000,
            "north_velocity": u_res['uy']*1000,
            "east_sigma": error_e*1000,
            "north_sigma": error_n*1000,
            "correlation_EN": 0.0,        
        }
    )
    df_res_h.head()
    df_res_h_dum = df_res_h.iloc[:, :4]

    df_res_v = pd.DataFrame(
        data={
            "x": df['lon'],
            "y": df['lat'],
            "east_velocity": 0.0,
            "north_velocity": u_res['uz']*1000,   
            "east_sigma": error_v*1000,
            "north_sigma": error_v*1000,
            "correlation_EN": 0.0,     
        }
    )
    df_res_v_dum = df_res_v.iloc[:, :4]

    return df_pred_h, df_pred_v, df_res_h, df_res_v, df_res_h_dum, df_res_v_dum


# buil observation disp. vectors for various structure models
df_pred_h_homhom, df_pred_v_homhom, df_res_h_homhom, df_res_v_homhom, \
    df_res_h_dum_homhom, df_res_v_dum_homhom = build_pred_disp_vector(df, u_pred_homhom, u_res_homhom, error_e, error_n, error_v)

df_pred_h_swhom, df_pred_v_swhom, df_res_h_swhom, df_res_v_swhom, \
    df_res_h_dum_swhom, df_res_v_dum_swhom = build_pred_disp_vector(df, u_pred_swhom, u_res_swhom, error_e, error_n, error_v)

df_pred_h_swsw, df_pred_v_swsw, df_res_h_swsw, df_res_v_swsw, \
    df_res_h_dum_swsw, df_res_v_dum_swsw = build_pred_disp_vector(df, u_pred_swsw, u_res_swsw, error_e, error_n, error_v)

df_pred_h_3dhom, df_pred_v_3dhom, df_res_h_3dhom, df_res_v_3dhom, \
    df_res_h_dum_3dhom, df_res_v_dum_3dhom = build_pred_disp_vector(df, u_pred_3dhom, u_res_3dhom, error_e, error_n, error_v)

df_pred_h_3d3d, df_pred_v_3d3d, df_res_h_3d3d, df_res_v_3d3d, \
    df_res_h_dum_3d3d, df_res_v_dum_3d3d = build_pred_disp_vector(df, u_pred_3d3d, u_res_3d3d, error_e, error_n, error_v)


## For inversion under same respective models


In [ ]:
def plot_hor_disp_obs_pred_res(df_obs_h, df_pred_h, df_res_h, df_res_h_dum, u_true, u_pred, 
                               scale_vec, color, lgdstr, region, errorbar=True):
    # plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.8/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")            
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_h, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e"+str(0.8/scale_vec)+"/0.39", 
                     vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
            # fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_h, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            else:
                fig.velo(data=df_res_h_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0") 
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                    "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")

    fig.show()

uh_max = np.hypot(df_obs_h_hom['east_velocity'].to_numpy(), df_obs_h_hom['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

## data misfit by under forward/inversion using respective models
plot_hor_disp_obs_pred_res(df_obs_h_hom, df_pred_h_homhom, df_res_h_homhom, df_res_h_dum_homhom, 
                           u_true_hom, u_pred_homhom, scale_vec, 'red', 'hom', region, errorbar=False)
plot_hor_disp_obs_pred_res(df_obs_h_sw, df_pred_h_swsw, df_res_h_swsw, df_res_h_dum_swsw,
                           u_true_sw, u_pred_swsw, scale_vec, 'red', 'SW', region, errorbar=False)
plot_hor_disp_obs_pred_res(df_obs_h_3d, df_pred_h_3d3d, df_res_h_3d3d, df_res_h_dum_3d3d, 
                           u_true_3d, u_pred_3d3d, scale_vec, 'red', '3d', region, errorbar=False)
# plot_hor_disp_obs_pred_res(df_obs_h_sw, df_pred_h_swhom, df_res_h_swhom, df_res_h_dum_swhom, 
#                            'blue', 'hom', region, errorbar=True)

In [ ]:
def plot_vert_disp_obs_pred_res(df_obs_v, df_pred_v, df_res_v, df_res_v_dum, u_true, u_pred, 
                                scale_vec, color, lgdstr, region, errorbar=True):
    # plot the vertical predicted displacements vs. observed at GNSS stations, and the residual
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.8/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")            
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_v, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e"+str(0.8/scale_vec)+"/0.39", 
                     vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.8/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color}+ea+g{color}+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Pred {lgdstr}", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")
            # fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Residual  
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            if errorbar:
                fig.velo(data=df_res_v, pen=f"0.5p,{color}", uncertaintyfill=None, line=f"0.25p,{color}", 
                        spec="e0.16/0.39", vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            else:
                fig.velo(data=df_res_v_dum, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.16/0.39", 
                        vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm   
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.25, font="9p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            if errorbar:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1],
                                        "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            else:
                lgdres = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.35, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen=f"0.5p,{color}", line=f"0.25p,{color}", spec="e0.8/0.39", 
                     vector=f"0.1c+a60+p1p,{color}+ea+g{color}")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res {lgdstr}", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            if errorbar:
                fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")
            else:
                fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.3, font="8p,Helvetica,black", justify="ML")

    fig.show()

uv_max = df_obs_v_hom['north_velocity'].to_numpy().max()
n = np.ceil(uv_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
scale_vec = 10    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uv_max, scale_vec)

## data misfit by under forward/inversion using respective models
plot_vert_disp_obs_pred_res(df_obs_v_hom, df_pred_v_homhom, df_res_v_homhom, df_res_v_dum_homhom, 
                            u_true_hom, u_pred_homhom, scale_vec, 'red', 'het', region, errorbar=False)
plot_vert_disp_obs_pred_res(df_obs_v_sw, df_pred_v_swsw, df_res_v_swsw, df_res_v_dum_swsw, 
                            u_true_sw, u_pred_swsw, scale_vec, 'red', 'het', region, errorbar=False)
plot_vert_disp_obs_pred_res(df_obs_v_3d, df_pred_v_3d3d, df_res_v_3d3d, df_res_v_dum_3d3d, 
                            u_true_3d, u_pred_3d3d, scale_vec, 'red', 'het', region, errorbar=False)
# plot_vert_disp_obs_pred_res(df_obs_v_sw, df_pred_v_swhom, df_res_v_swhom, df_res_v_dum_swhom, 
#                            'blue', 'hom', region, errorbar=False)

In [ ]:
def plot_disp_res_comparison(df_res_h_hom, df_res_v_hom, df_res_h_sw, df_res_v_sw, df_res_h_3d, df_res_v_3d,
                            u_true_hom, u_pred_hom, u_true_sw, u_pred_sw, u_true_3d, u_pred_3d,
                            color, region):
    
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")

        # Horizontal Residual  
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_h_sw, pen=f"0.5p,{color[1]}", line=f"0.25p,{color[1]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[1]}+ea+g{color[1]}+h0") 
            fig.velo(data=df_res_h_3d, pen=f"0.5p,{color[2]}", line=f"0.25p,{color[2]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[2]}+ea+g{color[2]}+h0")                         
            fig.velo(data=df_res_h_hom, pen=f"0.5p,{color[0]}", line=f"0.25p,{color[0]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[0]}+ea+g{color[0]}+h0") 
            rmse_hom = utp.rmse_3d_dataframe(u_true_hom, u_pred_hom)* 1000  # Convert to mm   
            rmse_sw = utp.rmse_3d_dataframe(u_true_sw, u_pred_sw)* 1000  # Convert to mm   
            rmse_3d = utp.rmse_3d_dataframe(u_true_3d, u_pred_3d)* 1000  # Convert to mm 
            fig.text(text=f"RMSE:", x=region[0]+0.1, y=region[-1]-0.15, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Hom: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.3, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"SW: {rmse_sw:.2f} mm", x=region[0]+0.1, y=region[-1]-0.45, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"3D: {rmse_3d:.2f} mm", x=region[0]+0.1, y=region[-1]-0.6, font="8p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.65, "east_velocity": [1], "north_velocity": [0]})
            lgdres_sw = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0]})
            lgdres_3d = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.35, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres_hom, pen=f"0.5p,{color[0]}", line=f"0.25p,{color[0]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[0]}+ea+g{color[0]}+h0")
            fig.velo(data=lgdres_sw, pen=f"0.5p,{color[1]}", line=f"0.25p,{color[1]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[1]}+ea+g{color[1]}+h0")
            fig.velo(data=lgdres_3d, pen=f"0.5p,{color[2]}", line=f"0.25p,{color[2]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[2]}+ea+g{color[2]}+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res Hom", x=region[0]+0.6, y=region[-2]+0.65, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res SW", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res 3D", x=region[0]+0.6, y=region[-2]+0.35, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Vertical Residual  
        with fig.set_panel(panel=[0, 1]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_v_sw, pen=f"0.5p,{color[1]}", line=f"0.25p,{color[1]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[1]}+ea+g{color[1]}+h0") 
            fig.velo(data=df_res_v_3d, pen=f"0.5p,{color[2]}", line=f"0.25p,{color[2]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[2]}+ea+g{color[2]}+h0")                         
            fig.velo(data=df_res_v_hom, pen=f"0.5p,{color[0]}", line=f"0.25p,{color[0]}", spec="e0.16/0.39", 
                    vector=f"0.1c+a45+p1p,{color[0]}+ea+g{color[0]}+h0") 
            rmse_hom = utp.rmse_3d_dataframe(u_true_hom, u_pred_hom)* 1000  # Convert to mm   
            rmse_sw = utp.rmse_3d_dataframe(u_true_sw, u_pred_sw)* 1000  # Convert to mm   
            rmse_3d = utp.rmse_3d_dataframe(u_true_3d, u_pred_3d)* 1000  # Convert to mm 
            fig.text(text=f"RMSE:", x=region[0]+0.1, y=region[-1]-0.15, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Hom: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.3, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"SW: {rmse_sw:.2f} mm", x=region[0]+0.1, y=region[-1]-0.45, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"3D: {rmse_3d:.2f} mm", x=region[0]+0.1, y=region[-1]-0.6, font="8p,Helvetica,black", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.35, "y": region[-2]+0.4, "east_velocity": [0], "north_velocity": [1]})
            lgdres_sw = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1]})
            lgdres_3d = pd.DataFrame(data={"x": region[0]+0.45, "y": region[-2]+0.2, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres_hom, pen=f"0.5p,{color[0]}", line=f"0.25p,{color[0]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[0]}+ea+g{color[0]}+h0")
            fig.velo(data=lgdres_sw, pen=f"0.5p,{color[1]}", line=f"0.25p,{color[1]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[1]}+ea+g{color[1]}+h0")
            fig.velo(data=lgdres_3d, pen=f"0.5p,{color[2]}", line=f"0.25p,{color[2]}", spec="e0.8/0.39", 
                     vector=f"0.1c+a45+p1p,{color[2]}+ea+g{color[2]}+h0")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res Hom", x=region[0]+0.6, y=region[-2]+0.65, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res SW", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=f"Res 3D", x=region[0]+0.6, y=region[-2]+0.35, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    fig.show()

color = ['lightgreen', 'red', 'blue']                
plot_disp_res_comparison(df_res_h_dum_homhom, df_res_v_dum_homhom, df_res_h_dum_swsw, df_res_v_dum_swsw, 
                         df_res_h_dum_3d3d, df_res_v_dum_3d3d, u_true_hom, u_pred_homhom, u_true_sw, u_pred_swsw,
                         u_true_3d, u_pred_3d3d, color, region)


## For inversion under correct model or ignoring the model


In [ ]:
# plot the predicted displacements vs. observed at GNSS stations, and the residual for ALL components
def plot_disp_obs_pred_res(df_obs_h, df_pred_hom_h, df_pred_h, df_res_hom_h_dum, df_res_h_dum, 
                           df_obs_v, df_pred_hom_v, df_pred_v, df_res_hom_v_dum, df_res_v_dum,
                           u_true, u_pred_hom, u_pred,  
                           region, struc_str_for, flag_savefig=False):
    
    ### Plot the comparison on horizontal components
    fig = pygmt.Figure()
    with fig.subplot(nrows=2, ncols=3, figsize=("18c", "12c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):
        
        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="5p", FONT_ANNOT_SECONDARY="10p")
        
        # homogeneous Observed vs. predicted
        with fig.set_panel(panel=[0, 0]):
            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_hom_h, pen="0.5p,blue", line="0.25p,blue", spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")        

        # heterogeneous Observed vs. predicted
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_h, pen="0.5p,red", line="0.25p,red", spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # residuals, hom vs het
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_hom_h_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_h_dum, pen="0.5p,red", line="0.25p,red", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,red+ea+gred")
            
            rmse_hom = utp.rmse_3d_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm   
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm  
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2, font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4, font="9p,Helvetica,red", justify="ML")

            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        ### Plot the comparison on vectical components
        # homogeneous Observed vs. predicted
        with fig.set_panel(panel=[1, 0]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_hom_v, pen="0.5p,blue", line="0.25p,blue", spec="e0.05/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,blue", line="0.25p,blue", spec="e0.5/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred hom", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")      

        # heterogeneous Observed vs. predicted
        with fig.set_panel(panel=[1, 1]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            #observed displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e0.05/0.39", 
                    vector="0.1c+a60+p1p,black+ea+gblack")
            # predicted disp. from inferred slip
            fig.velo(data=df_pred_v, pen="0.5p,red", line="0.25p,red", spec="e0.05/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                                "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a60+p1p,black+ea+gblack")
            lgdpred = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdpred, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.9, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Pred het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # residuals, hom vs het
        with fig.set_panel(panel=[1, 2]):
            fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                    area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            fig.grdcontour(grid=grd_file, levels=5, limit="-100/0", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # disp. residual == observed - predicted
            fig.velo(data=df_res_hom_v_dum, pen="0.5p,blue", line="0.25p,blue", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,blue+ea+gblue")
            fig.velo(data=df_res_v_dum, pen="0.5p,red", line="0.25p,red", spec="e0.16/0.39", 
                    vector="0.1c+a60+p1p,red+ea+gred")
            rmse_hom = utp.rmse_3d_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm   
            rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm  
            fig.text(text=f"RMSE: {rmse_hom:.2f} mm", x=region[0]+0.1, y=region[-1]-0.2, font="9p,Helvetica,blue", justify="ML")
            fig.text(text=f"RMSE: {rmse:.2f} mm", x=region[0]+0.1, y=region[-1]-0.4, font="9p,Helvetica,red", justify="ML")
            # plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.9, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdres_hom = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.45, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres_hom, pen="0.5p,blue", line="0.25p,blue", spec="e0.8/0.39", vector="0.1c+a60+p1p,blue+ea+gblue")
            lgdres = pd.DataFrame(data={"x": region[0]+0.5, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
            fig.velo(data=lgdres, pen="0.5p,red", line="0.25p,red", spec="e0.8/0.39", vector="0.1c+a60+p1p,red+ea+gred")
            # plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res hom", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Res het", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
            fig.text(text="5 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_datafit.pdf")

## data misfit under inversion using hom or het models based on same data from het models
uh_max = np.hypot(df_obs_h_sw['east_velocity'].to_numpy(), df_obs_h_sw['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)
plot_disp_obs_pred_res(df_obs_h_sw, df_pred_h_swhom, df_pred_h_swsw, df_res_h_dum_swhom, df_res_h_dum_swsw, 
                       df_obs_v_sw, df_pred_v_swhom, df_pred_v_swsw, df_res_v_dum_swhom, df_res_v_dum_swsw,
                       u_true_sw, u_pred_swhom, u_pred_swsw, 
                       region, sw_str, flag_savefig=False)

plot_disp_obs_pred_res(df_obs_h_3d, df_pred_h_3dhom, df_pred_h_3d3d, df_res_h_dum_3dhom, df_res_h_dum_3d3d, 
                       df_obs_v_3d, df_pred_v_3dhom, df_pred_v_3d3d, df_res_v_dum_3dhom, df_res_v_dum_3d3d,
                       u_true_3d, u_pred_3dhom, u_pred_3d3d, 
                       region, het3d_str, flag_savefig=False)


In [ ]:
# def plot_res_hist(u_true, u_pred_hom, u_res_hom, u_pred=None, u_res=None, struc_str_for=None, flag_savefig=False):

#     # plot the histogram of difference in residuals
#     if struc_str_for == homo_str:
#         fig, axes = plt.subplots(1, 1, figsize=(4, 4.5), dpi=300)
#         bin_width = 0.25

#         ax = axes

#         xmin, xmax = 0, 5 
#         bins = np.arange(xmin, xmax + bin_width, bin_width)
#         counts1, _, _ = ax.hist(u_res_hom['mag']*1000, bins=bins, color='blue', alpha=0.7, edgecolor='black',label='Homogeneous')

#         ymin, ymax = 0, np.max(counts1) + 4

#         median_hom = np.median(u_res_hom['mag']*1000)
#         sigma_hom = np.std(u_res_hom['mag']*1000)

#         rmse_hom = utp.rmse_3d_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm
#         print(f"HOM Residual RMSE: {rmse_hom:.2f} mm")
#         print(f"HOM Residual median: {median_hom:.2f} mm, sigma: {sigma_hom:.2f} mm")

#         ax.plot([median_hom, median_hom], [ymin, ymax], '--', color='blue', label=fr'Hom median={median_hom:.1f}')
#         ax.errorbar(median_hom, ymax-2, xerr=sigma_hom, fmt='o', color='blue', capsize=5, capthick=2, 
#                     linewidth=2, label=fr'Hom $\pm1\sigma=\pm{sigma_hom:.1f}$')
#         ax.text(0.5, 0.9, f"Hom Res RMSE:{rmse_hom:.2f}", fontsize=9, color='k', transform=ax.transAxes)

#         ax.legend(fontsize=9, loc='right')
#         ax.set_xlabel('Residual (mm)')
#         ax.set_ylabel('Frequency')
#         ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
#         ax.set_ylim(ymin, ymax)
#         ax.set_yticks(np.arange(ymin, ymax+2, 2))
#         ax.set_xlim(xmin, xmax)

#     else:
#         fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.5), dpi=300)

#         bin_width = 0.25

#         ax = axes[0]

#         xmin, xmax = 0, 5 
#         bins = np.arange(xmin, xmax + bin_width, bin_width)
#         counts1, _, _ = ax.hist(u_res_hom['mag']*1000, bins=bins, color='blue', alpha=0.7, edgecolor='black',label='Homogeneous')
#         counts2, _, _ = ax.hist(u_res['mag']*1000, bins=bins, color='red', alpha=0.7, edgecolor='black',label='Heterogeneous')

#         ymin, ymax = 0, max(np.max(counts1), np.max(counts2)) + 4

#         median_hom = np.median(u_res_hom['mag']*1000)
#         sigma_hom = np.std(u_res_hom['mag']*1000)
#         median = np.median(u_res['mag']*1000)
#         sigma = np.std(u_res['mag']*1000)

#         rmse_hom = utp.rmse_3d_dataframe(u_pred_hom, u_true)* 1000  # Convert to mm
#         print(f"HOM Residual RMSE: {rmse_hom:.2f} mm")
#         print(f"HOM Residual median: {median_hom:.2f} mm, sigma: {sigma_hom:.2f} mm")

#         rmse = utp.rmse_3d_dataframe(u_pred, u_true)* 1000  # Convert to mm
#         print(f"HET Residual RMSE: {rmse:.2f} mm")
#         print(f"HET Residual median: {median:.2f} mm, sigma: {sigma:.2f} mm")

#         ax.plot([median_hom, median_hom], [ymin, ymax], '--', color='blue', label=fr'Hom median={median_hom:.1f}')
#         ax.plot([median, median], [ymin, ymax], '--', color='red', label=fr'Het median={median:.1f}')
#         ax.errorbar(median_hom, ymax-2, xerr=sigma_hom, fmt='o', color='blue', capsize=5, capthick=2, 
#                     linewidth=2, label=fr'Hom $\pm1\sigma=\pm{sigma_hom:.1f}$')
#         ax.errorbar(median, ymax-2, xerr=sigma, fmt='o', color='red', capsize=5, capthick=2, 
#                     linewidth=2, label=fr'Het $\pm1\sigma=\pm{sigma:.1f}$')
#         ax.text(0.5, 0.9, f"Hom Res RMSE:{rmse_hom:.2f}", fontsize=9, color='k', transform=ax.transAxes)
#         ax.text(0.5, 0.8, f"Het Res RMSE:{rmse:.2f}", fontsize=9, color='k', transform=ax.transAxes)

#         ax.legend(fontsize=9, loc='right')
#         ax.set_xlabel('Residual (mm)')
#         ax.set_ylabel('Frequency')
#         ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
#         ax.set_ylim(ymin, ymax)
#         ax.set_yticks(np.arange(ymin, ymax+2, 2))
#         ax.set_xlim(xmin, xmax)

#         ax = axes[1]
#         xmin, xmax = -2, 2
#         bins = np.arange(xmin - bin_width, xmax + bin_width, bin_width)
#         counts1, _, _ = ax.hist(u_res['mag']*1000-u_res_hom['mag']*1000, bins=bins, color='dimgray',alpha=0.7, edgecolor='black',label='het-hom')
#         ymin, ymax = 0, np.max(counts1) + 2

#         median = np.median(u_res['mag']*1000-u_res_hom['mag']*1000)
#         ax.plot([median, median], [ymin, ymax], '--', color='black', label=fr'median={median:.2f}')

#         ax.legend(fontsize=9, loc='right')
#         ax.set_xlabel('Difference (mm)')
#         ax.set_ylabel('Frequency')
#         ax.set_title('Residual difference (Het-Hom)', fontweight='bold')
#         ax.set_ylim(ymin, ymax)
#         ax.set_yticks(np.arange(ymin, ymax+2, 2))
#         ax.set_xlim(xmin, xmax)

#     if flag_savefig:
#         fig.savefig(figpath + meshname + struc_str_for + "_res.pdf")


# plot_res_hist(u_true_hom, u_pred_homhom, u_res_homhom, struc_str_for=homo_str, flag_savefig=False)       
# plot_res_hist(u_true_sw, u_pred_swhom, u_res_swhom, u_pred=u_pred_swsw, u_res=u_res_swsw, 
#               struc_str_for=sw_str, flag_savefig=False)        
# plot_res_hist(u_true_3d, u_pred_3dhom, u_res_3dhom, u_pred=u_pred_3d3d, u_res=u_res_3d3d, 
#               struc_str_for=het3d_str, flag_savefig=False)     



In [ ]:
print("L2-norm:") 

resl2homhom = np.linalg.norm(u_res_homhom[['ux','uy','uz']].values)
print("Hom-Hom:{0:.6e}".format(resl2homhom))

resl2swhom = np.linalg.norm(u_res_swhom[['ux','uy','uz']].values)
resl2swsw = np.linalg.norm(u_res_swsw[['ux','uy','uz']].values)
print("SW-Hom:{0:.6e}; SW-SW:{1:.6e}".format(resl2swhom, resl2swsw) )

resl23dhom = np.linalg.norm(u_res_3dhom[['ux','uy','uz']].values)
resl23d3d = np.linalg.norm(u_res_3d3d[['ux','uy','uz']].values)
print("3d-Hom:{0:.6e}; 3d-3d:{1:.6e}".format(resl23dhom, resl23d3d) )
